# 데이터 파이프라인 종합 과제

## "서울 지하철 실시간 혼잡도 모니터링 시스템"

---

## 과제 개요

여러분은 서울시 스마트시티 프로젝트의 **데이터 엔지니어**입니다.
시민들에게 지하철 혼잡도를 실시간으로 제공하여 쾌적한 출퇴근을 돕고자 합니다.

이 과제에서는 **가장 단순한 형태의 데이터 처리**에서 시작하여,
**점진적으로 발생하는 문제들을 해결**하면서 완전한 데이터 파이프라인을 구축합니다.

> **핵심 철학**: "왜 이 도구가 필요한가?"를 직접 체감하며 배웁니다.
> Kafka, Spark 같은 도구들이 어떤 문제를 해결하는지 몸으로 느껴보세요.

---

## 학습 목표

이 과제를 마치면 다음을 할 수 있습니다:

### 1. 데이터 파이프라인 이해
- 데이터 파이프라인의 핵심 구성요소(Source → Processing → Sink)를 설명할 수 있다
- **OLTP vs OLAP** 시스템의 차이를 이해하고 구분할 수 있다
- **ETL vs ELT** 패턴의 차이와 각각의 적합한 상황을 설명할 수 있다
- **배치 처리 vs 스트림 처리**의 차이를 코드로 체감한다

### 2. 도구별 역할 이해 (왜 필요한가?)
- **Docker**: 환경 불일치 문제를 해결
- **Kafka**: 데이터 유실과 시스템 결합도 문제를 해결
- **Spark**: 단일 노드의 처리 한계를 해결
- **GitHub Actions**: 수동 실행의 피로를 해결
- **S3**: 로컬 스토리지의 확장성 한계를 해결

### 3. 다음 학습과의 연결
- 이 과제에서 구축한 파이프라인은 다음 수업의 **Elasticsearch + Kibana**와 자연스럽게 연결됩니다
- 파이프라인으로 수집한 데이터를 검색 엔진에 저장하고 대시보드로 시각화하는 흐름을 경험하게 됩니다

---

## 데이터 파이프라인이란?

### 핵심 개념

데이터 파이프라인은 데이터가 **원천(Source)** 에서 **목적지(Sink)** 까지
흐르는 일련의 과정입니다.

**Source (원천)** → **Processing (처리)** → **Sink (목적지)**

| 단계 | 역할 | 예시 |
|------|------|------|
| **Source** | 데이터 발생 | 데이터베이스, API, 파일, IoT 센서 |
| **Processing** | 데이터 가공 | 필터링/변환, 집계/조인, 정제/검증, 머신러닝 |
| **Sink** | 데이터 저장/활용 | 데이터 웨어하우스, 데이터 레이크, 대시보드, 알림 시스템 |

### 이 과제에서 구축할 파이프라인

**서울 지하철 혼잡도 모니터링 파이프라인**

| 단계 | 구성요소 | 설명 |
|------|----------|------|
| **Source** | 서울시 공공API | 실시간 지하철 도착 정보 |
| **Ingestion** | Kafka (메시지큐) | 데이터 수집 및 버퍼링 |
| **Processing** | Spark (스트림) | 실시간 데이터 처리 |
| **Storage** | S3 (데이터레이크) | 원본 데이터 저장 |
| **Analytics** | Athena / 로컬 분석 / ES+Kibana | SQL 분석 및 시각화 |

---

## OLTP vs OLAP

### 두 시스템의 근본적 차이

데이터베이스 시스템은 **목적**에 따라 두 가지로 구분됩니다.
이 차이를 이해하는 것이 데이터 파이프라인을 이해하는 첫걸음입니다.

**"거래를 기록하는 시스템" vs "분석을 위한 시스템"**

| 시스템 | 풀네임 | 핵심 질문 |
|--------|--------|-----------|
| **OLTP** | Online Transaction Processing | "지금 이 순간의 거래" (예: 승객이 지하철 탑승 → 탑승 기록 INSERT) |
| **OLAP** | Online Analytical Processing | "과거 데이터의 패턴 분석" (예: 이번 달 혼잡한 시간대는? → 30일치 데이터 집계) |

### 상세 비교

| 구분 | OLTP | OLAP |
|------|------|------|
| **풀네임** | Online Transaction Processing | Online Analytical Processing |
| **목적** | 실시간 트랜잭션 처리 | 분석 및 리포팅 |
| **주요 연산** | INSERT, UPDATE, DELETE | SELECT (복잡한 집계) |
| **쿼리 특징** | 단순, 빠름 (1행 조회) | 복잡, 느림 (수억 행 스캔) |
| **데이터 양** | 최신 상태의 적은 양 | 대량의 히스토리 데이터 |
| **정규화** | 높음 (3NF) | 낮음 (비정규화, 스타 스키마) |
| **동시 사용자** | 수천~수만 명 | 수십~수백 명 |
| **응답 시간** | 밀리초 | 초~분 |
| **사용자** | 고객, 운영팀 | 데이터 분석가, 경영진 |
| **예시** | MySQL, PostgreSQL | Redshift, BigQuery, Snowflake |

### 실제 예시로 이해하기

```python
# OLTP 쿼리 예시 - 특정 승객의 현재 탑승 정보 조회
# 특징: 단순, 빠름, 1행 반환
"""
SELECT station_name, boarding_time
FROM boarding_log
WHERE passenger_id = 'P12345'
  AND boarding_time >= NOW() - INTERVAL 1 HOUR;
"""

# OLAP 쿼리 예시 - 월별 시간대별 혼잡도 분석
# 특징: 복잡, 수억 행 스캔, 집계
"""
SELECT
    DATE_TRUNC('month', boarding_time) as month,
    EXTRACT(HOUR FROM boarding_time) as hour,
    line_name,
    AVG(congestion_level) as avg_congestion,
    COUNT(*) as total_boardings
FROM boarding_log
WHERE boarding_time >= '2024-01-01'
GROUP BY 1, 2, 3
ORDER BY avg_congestion DESC;
"""
```

### 왜 둘 다 필요한가?

**데이터 흐름**: 실시간 서비스 → OLTP (운영DB) → Data Pipeline (ETL/ELT) → OLAP (분석 DW) → 분석/리포팅

| 흐름 | 예시 |
|------|------|
| OLTP에서 발생 | "승객 탑승 기록", "실시간 위치 조회" |
| 파이프라인 전달 | ETL / ELT 처리 |
| OLAP에서 분석 | "월간 혼잡도 리포트", "노선별 이용 패턴 분석" |

> **핵심**: OLTP에서 운영 데이터를 수집하고, 파이프라인을 통해 OLAP으로 보내 분석합니다.
> 이것이 바로 **데이터 파이프라인**의 역할입니다.

---

## ETL vs ELT

### 데이터를 옮기는 두 가지 패턴

데이터 파이프라인에서 데이터를 Source에서 Sink로 옮길 때 두 가지 패턴이 있습니다.

**ETL (Extract → Transform → Load)**

- 순서: Source → Extract → **Transform (중간 서버에서 변환)** → Load → Target
- 특징: 변환 후 적재 → 깨끗한 데이터만 저장
- 적합: 데이터 웨어하우스, 정형화된 분석

**ELT (Extract → Load → Transform)**

- 순서: Source → Extract → Load (원본 그대로 저장) → **Transform (Target에서 변환)** → 분석
- 특징: 일단 적재 후 변환 → 원본 데이터 보존
- 적합: 데이터 레이크, 유연한 분석, 클라우드 환경

### 상세 비교

| 구분 | ETL | ELT |
|------|-----|-----|
| **변환 시점** | Load 전 (중간 서버) | Load 후 (Target 시스템) |
| **원본 데이터** | 보존 안 됨 | 보존됨 |
| **유연성** | 낮음 (스키마 변경 어려움) | 높음 (필요시 재변환) |
| **처리 비용** | 중간 서버 비용 | Target 시스템 비용 |
| **적합한 상황** | 정형화된 분석, 규제 환경 | 탐색적 분석, 빅데이터 |
| **대표 도구** | Spark, Airflow | dbt, Redshift, Airbyte |

### 이 과제에서의 적용

우리 파이프라인은 **ELT 패턴**을 사용합니다:

**서울시 API** → Extract (API호출) → Load (저장) → **S3 (원본)** → Transform (Spark/Athena) → **분석**

**왜 ELT인가?**
1. 원본 데이터를 S3에 보존 → 나중에 다른 분석 가능
2. 변환 로직을 쉽게 수정 가능
3. Spark, Athena 등 강력한 처리 엔진 활용

---

## 배치 처리 vs 스트림 처리

### 데이터 처리 타이밍의 차이

**배치(Batch) 처리**

- 방식: 데이터를 일정 기간 모아서 한꺼번에 처리
- 예: 매일 자정에 어제 데이터 분석, 매주 월요일 주간 리포트
- 장점: 구현 단순, 대용량 효율적
- 단점: 지연(Latency) 발생

**스트림(Stream) 처리**

- 방식: 데이터가 들어오는 즉시 처리
- 예: 실시간 혼잡도 알림, 실시간 이상 탐지
- 장점: 낮은 지연, 실시간 인사이트
- 단점: 구현 복잡, 장애 복구 어려움

### 실제 비유로 이해하기

| 처리 방식 | 비유 | 설명 |
|-----------|------|------|
| **배치 처리** | 우편 배달 | 아침에 편지들을 모아서 오후에 한번에 배달. 효율적이지만 긴급한 편지도 기다려야 함 |
| **스트림 처리** | 카카오톡 | 메시지 보내면 즉시 상대방에게 전달. 빠르지만 시스템이 더 복잡함 |

### 이 과제에서 둘 다 경험합니다

| Phase | 처리 방식 | 사용 도구 | 시나리오 |
|-------|----------|----------|----------|
| Phase 1-2 | 배치 | Python, Pandas | 1시간마다 데이터 수집 후 분석 |
| Phase 3-4 | 스트림 | Kafka, Spark Streaming | 실시간 혼잡도 알림 |
| Phase 5 | 배치 (스케줄) | GitHub Actions | 매시간 자동 실행 |

---

## 진행 방식

### 페어 프로그래밍

2인 1조로 진행합니다. 각 Phase마다 **진행자(Driver)** 와 **관찰자(Navigator)** 역할을
번갈아가며 수행합니다.

**페어 프로그래밍 구조**

| 역할 | 담당 업무 |
|------|-----------|
| **진행자 (Driver)** | 키보드 조작, 코드 작성, 명령어 실행 |
| **관찰자 (Navigator)** | 방향 제시, 문서/힌트 확인, 오류 발견 |

두 역할은 **소통**하며 협업합니다.

> **Tip**: 막히면 역할을 바꿔보세요!
> **Tip**: Phase 끝날 때마다 역할을 교대하고 Git commit 하세요

### Git 협업

- 하나의 저장소를 공유하며 작업합니다
- 각 Phase 완료 시 commit 메시지에 진행자 이름을 포함합니다
- 예: `[Phase 1] 기본 파이프라인 구현 - Driver: 홍길동`

---

## 과제 시나리오

### 서울시 스마트시티 프로젝트 상황

**현재 상황**:
- 서울 지하철은 하루 약 700만 명이 이용
- 출퇴근 시간 특정 노선/역의 혼잡도가 매우 높음
- 시민들은 "지금 몇 호선이 덜 붐빌까?"를 알고 싶어함
- 서울시는 공공 API로 실시간 지하철 정보를 제공 중

**요구사항**:
1. 실시간 지하철 도착 정보를 수집하여 혼잡도 분석
2. 시간대별, 노선별 혼잡도 패턴 파악
3. 점차 데이터 양이 늘어나도 처리 가능해야 함
4. 최종적으로 Kibana 대시보드로 시각화 (다음 수업 연계)

### 사용할 API

**서울시 열린데이터광장 - 서울시 지하철 실시간 도착정보**

```
API 엔드포인트:
http://swopenAPI.seoul.go.kr/api/subway/{KEY}/json/realtimeStationArrival/0/5/{역이름}

응답 예시:
{
  "realtimeArrivalList": [
    {
      "subwayId": "1002",           // 호선 (1002 = 2호선)
      "statnNm": "강남",             // 역 이름
      "trainLineNm": "성수행 - 강남방면", // 열차 방향
      "arvlMsg2": "전역 도착",        // 도착 메시지
      "arvlMsg3": "강남",            // 다음역
      "barvlDt": "0",               // 도착 예정 시간(초)
      "recptnDt": "2024-01-15 08:30:15" // 수신 시간
    }
  ]
}
```

> **API 키 발급**: https://data.seoul.go.kr/ 에서 회원가입 후 API 키 발급
> (수업용 공용 키를 제공할 수도 있습니다)

### 분석 목표

1. **실시간 도착 정보 수집**: 주요 역의 실시간 도착 정보 수집
2. **시간대별 패턴 분석**: 시간대별 열차 배차 간격 분석
3. **노선별 비교**: 호선별 평균 대기 시간 비교
4. **혼잡 역 탐지**: 배차 간격이 긴 역 = 혼잡 예상 역

---

## Phase 구성

이 과제는 6개의 Phase로 구성됩니다. 각 Phase는 이전 Phase의 **한계를 극복**하는
방식으로 진행됩니다.

### Phase 1-3: 기초부터 메시지 큐까지

| Phase | 주제 | 데이터 규모 | 체감할 한계 | 해결 도구 |
|-------|------|------------|------------|----------|
| **1** | 단순 파이프라인 | 100건 | - | Python |
| **2** | Docker 컨테이너화 | 100건 | 환경 불일치 | Docker |
| **3** | Kafka 메시지 큐 | 10,000건 | 데이터 유실, 결합도 | Kafka |

### Phase 4-6: 분산 처리부터 클라우드까지

| Phase | 주제 | 데이터 규모 | 체감할 한계 | 해결 도구 |
|-------|------|------------|------------|----------|
| **4** | Spark 분산 처리 | 100만건 | 처리 성능 | Spark |
| **5** | GitHub Actions | - | 수동 실행 | Actions |
| **6** | AWS 클라우드 연계 | - | 확장성 | S3, Athena |

### Phase별 복잡도 증가

| Phase | 도구 | 복잡도 |
|-------|------|--------|
| 1 | Python | ★ |
| 2 | Docker | ★★ |
| 3 | Kafka | ★★★ |
| 4 | Spark | ★★★★ |
| 5 | GitHub Actions | ★★★★★ |
| 6 | S3 + Athena | ★★★★★★ |

**Phase 1~3**: 기초 단계 / **Phase 4~6**: 심화 단계

### 다음 수업과의 연결

**이 과제에서 구축한 파이프라인**

API → Kafka → Spark → **S3**

↓

**다음 수업: Elasticsearch & Kibana**

S3/파일 데이터 → Elasticsearch 인덱싱 → Kibana 대시보드

→ 이 과제에서 수집한 데이터를 활용하여 시각화 대시보드 구축

---

## 환경 설정

### 필요한 도구

- WSL2 + Ubuntu 환경
- Docker Desktop 설치 완료
- VSCode + Docker Extension
- Git 설치 및 GitHub 계정
- AWS Academy 계정 (Phase 6용)
- 서울시 열린데이터광장 API 키

### 시작 전 확인사항

```bash
# Docker 확인
docker --version
docker compose version

# Git 확인
git --version

# Python 확인 (호스트)
python3 --version
```

### 저장소 초기화

```bash
# 팀 저장소 클론 (또는 새로 생성)
git clone <your-team-repo-url>
cd <repo-name>

# 또는 새 저장소 초기화
mkdir seoul-metro-pipeline
cd seoul-metro-pipeline
git init

# 기본 구조 생성
mkdir -p src docker .github/workflows data
touch .gitignore README.md
```

### .gitignore 설정

```bash
# .gitignore
__pycache__/
*.pyc
.env
*.log
data/*.json
data/*.parquet
.DS_Store
```

---

## 다음 단계

**Phase 1: 단순 파이프라인**으로 이동하세요.

> 파일: `01_Phase1_단순파이프라인.py`

Phase 1에서는 가장 기본적인 형태의 데이터 처리를 구현합니다.
Docker도 없이, Kafka도 없이, **순수 Python**만으로 시작합니다.

이 단순한 방식으로 시작하면서, 점차 "왜 이런 도구들이 필요한가?"를
직접 체감하게 될 것입니다.

---

## 참고 자료

- Docker 공식 문서: https://docs.docker.com/
- Kafka 공식 문서: https://kafka.apache.org/documentation/
- Spark 공식 문서: https://spark.apache.org/docs/latest/
- GitHub Actions 문서: https://docs.github.com/en/actions
- 서울 열린데이터광장: https://data.seoul.go.kr/

---


# Phase 1: 단순 파이프라인

## "가장 단순한 형태로 시작하기"

---

## 학습 목표

이 Phase를 마치면 다음을 할 수 있습니다:

- 공공 API를 호출하여 JSON 데이터를 받아올 수 있다
- Python으로 데이터를 파일에 저장하고 읽을 수 있다
- 간단한 데이터 분석(집계, 필터링)을 수행할 수 있다
- 이 단순한 방식의 **한계**를 직접 체감한다

---

## 이 Phase의 목표

**Phase 1: 가장 단순한 파이프라인**

**서울시 API** → **Python (requests)** → **JSON 파일 (저장)** → **Pandas (분석)**

| 항목 | 내용 |
|------|------|
| 도구 | Python, requests, pandas |
| 데이터 규모 | ~100건 |
| 처리 방식 | 배치 (수동 실행) |

---

## 역할 분담 (Phase 1)

| 역할 | 담당 |
|------|------|
| **진행자(Driver)** | 학생 A |
| **관찰자(Navigator)** | 학생 B |

> Phase가 끝나면 역할을 교대하고 Git commit 하세요!

---

## Step 1: 프로젝트 구조 만들기

### 과제 1-1: 디렉토리 구조 생성

터미널에서 다음 명령어를 실행하여 프로젝트 구조를 만드세요.

```bash
# 프로젝트 디렉토리 생성
mkdir -p seoul-metro-pipeline
cd seoul-metro-pipeline

# 하위 디렉토리 생성
mkdir -p src data output

# 빈 파일 생성
touch src/__init__.py
touch src/api_client.py
touch src/processor.py
touch .env.example
touch .gitignore
touch README.md
```

### 예상 결과

<details>
<summary>📁 디렉토리 구조 확인 (클릭해서 펼치기)</summary>

```
seoul-metro-pipeline/
├── src/
│   ├── __init__.py
│   ├── api_client.py      # API 호출 코드
│   └── processor.py       # 데이터 처리 코드
├── data/                  # 수집된 원본 데이터
├── output/                # 분석 결과
├── .env.example           # 환경변수 예시
├── .gitignore
└── README.md
```

</details>

---

## Step 2: API 키 설정

### 과제 1-2: 서울시 열린데이터광장 API 키 발급

1. https://data.seoul.go.kr/ 접속
2. 회원가입 또는 로그인
3. 마이페이지 → 인증키 신청
4. 발급받은 키를 `.env` 파일에 저장

```bash
# .env 파일 생성 (실제 키로 교체)
echo "SEOUL_API_KEY=your_api_key_here" > .env
```

### .env.example 작성

```bash
# .env.example (Git에 올라가는 예시 파일)
SEOUL_API_KEY=sample_key_replace_with_your_key
```

### .gitignore 작성

```bash
# .gitignore
.env
__pycache__/
*.pyc
data/*.json
output/*.csv
.DS_Store
```

<details>
<summary>💡 힌트: API 키 없이 테스트하기</summary>

서울시 API는 일부 공개 데이터의 경우 샘플 키를 제공합니다.
또는 강사가 제공하는 수업용 공용 키를 사용할 수 있습니다.

테스트용으로 `sample` 키를 사용해볼 수 있지만, 호출 횟수 제한이 있습니다.

</details>

---

## Step 3: API 클라이언트 구현

### 과제 1-3: 서울시 지하철 API 호출 코드 작성

`src/api_client.py` 파일을 작성하세요.

**요구사항:**
- 환경변수에서 API 키를 읽어온다
- 역 이름을 입력받아 실시간 도착 정보를 조회한다
- JSON 응답을 Python 딕셔너리로 반환한다
- 에러 처리를 포함한다

### 빈칸 채우기

```python
"""
서울시 지하철 실시간 도착정보 API 클라이언트
"""
import os
import requests
from dotenv import load_dotenv

# .env 파일에서 환경변수 로드
load_dotenv()


class SeoulMetroAPI:
    """서울시 지하철 API 클라이언트"""

    BASE_URL = "http://swopenAPI.seoul.go.kr/api/subway"

    def __init__(self):
        # TODO: 환경변수에서 API 키 가져오기
        self.api_key = ________________

        if not self.api_key:
            raise ValueError("SEOUL_API_KEY 환경변수가 설정되지 않았습니다.")

    def get_arrival_info(self, station_name: str) -> dict:
        """
        특정 역의 실시간 도착 정보를 조회합니다.

        Args:
            station_name: 역 이름 (예: "강남", "서울역")

        Returns:
            API 응답 딕셔너리
        """
        # TODO: API 엔드포인트 URL 완성하기
        # 형식: {BASE_URL}/{API_KEY}/json/realtimeStationArrival/0/10/{역이름}
        url = ________________

        try:
            # TODO: GET 요청 보내기
            response = ________________
            response.raise_for_status()

            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"API 호출 실패: {e}")
            return {}

    def get_multiple_stations(self, station_names: list) -> list:
        """
        여러 역의 도착 정보를 조회합니다.

        Args:
            station_names: 역 이름 리스트

        Returns:
            각 역의 응답을 담은 리스트
        """
        results = []

        for station in station_names:
            # TODO: 각 역의 도착 정보 조회하기
            data = ________________

            if data:
                results.append({
                    "station": station,
                    "data": data
                })

        return results


# 테스트 코드
if __name__ == "__main__":
    api = SeoulMetroAPI()

    # 강남역 도착 정보 조회
    result = api.get_arrival_info("강남")
    print(f"조회 결과 키: {result.keys()}")

    if "realtimeArrivalList" in result:
        arrivals = result["realtimeArrivalList"]
        print(f"도착 정보 개수: {len(arrivals)}")

        # 첫 번째 도착 정보 출력
        if arrivals:
            first = arrivals[0]
            print(f"역명: {first.get('statnNm')}")
            print(f"방향: {first.get('trainLineNm')}")
            print(f"도착메시지: {first.get('arvlMsg2')}")
```

<details>
<summary>💡 힌트 1: 환경변수 읽기</summary>

```python
# os.getenv() 또는 os.environ.get() 사용
self.api_key = os.getenv("SEOUL_API_KEY")
```

</details>

<details>
<summary>💡 힌트 2: URL 구성</summary>

```python
# f-string으로 URL 구성
url = f"{self.BASE_URL}/{self.api_key}/json/realtimeStationArrival/0/10/{station_name}"
```

</details>

<details>
<summary>💡 힌트 3: GET 요청</summary>

```python
response = requests.get(url)
```

</details>

<details>
<summary>✅ 모범 답안</summary>

```python
"""
서울시 지하철 실시간 도착정보 API 클라이언트
"""
import os
import requests
from dotenv import load_dotenv

load_dotenv()


class SeoulMetroAPI:
    """서울시 지하철 API 클라이언트"""

    BASE_URL = "http://swopenAPI.seoul.go.kr/api/subway"

    def __init__(self):
        self.api_key = os.getenv("SEOUL_API_KEY")

        if not self.api_key:
            raise ValueError("SEOUL_API_KEY 환경변수가 설정되지 않았습니다.")

    def get_arrival_info(self, station_name: str) -> dict:
        url = f"{self.BASE_URL}/{self.api_key}/json/realtimeStationArrival/0/10/{station_name}"

        try:
            response = requests.get(url)
            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"API 호출 실패: {e}")
            return {}

    def get_multiple_stations(self, station_names: list) -> list:
        results = []

        for station in station_names:
            data = self.get_arrival_info(station)

            if data:
                results.append({
                    "station": station,
                    "data": data
                })

        return results


if __name__ == "__main__":
    api = SeoulMetroAPI()
    result = api.get_arrival_info("강남")
    print(f"조회 결과 키: {result.keys()}")

    if "realtimeArrivalList" in result:
        arrivals = result["realtimeArrivalList"]
        print(f"도착 정보 개수: {len(arrivals)}")

        if arrivals:
            first = arrivals[0]
            print(f"역명: {first.get('statnNm')}")
            print(f"방향: {first.get('trainLineNm')}")
            print(f"도착메시지: {first.get('arvlMsg2')}")
```

</details>

---

## Step 4: 데이터 수집기 구현

### 과제 1-4: 여러 역의 데이터를 수집하고 파일로 저장

`src/collector.py` 파일을 작성하세요.

**요구사항:**
- 주요 역 목록을 정의한다 (예: 강남, 홍대입구, 신도림, 서울역, 잠실)
- 각 역의 도착 정보를 수집한다
- 수집 시간을 포함하여 JSON 파일로 저장한다
- 파일명은 `data/arrivals_YYYYMMDD_HHMMSS.json` 형식으로 한다

### 빈칸 채우기

```python
"""
지하철 도착 정보 수집기
"""
import json
from datetime import datetime
from pathlib import Path
from api_client import SeoulMetroAPI


# 수집할 주요 역 목록
TARGET_STATIONS = [
    "강남", "홍대입구", "신도림", "서울역", "잠실",
    "신림", "구로디지털단지", "여의도", "종로3가", "동대문"
]


def collect_arrivals(stations: list = None) -> dict:
    """
    여러 역의 도착 정보를 수집합니다.

    Args:
        stations: 수집할 역 목록 (None이면 TARGET_STATIONS 사용)

    Returns:
        수집된 데이터 딕셔너리
    """
    if stations is None:
        stations = TARGET_STATIONS

    api = SeoulMetroAPI()

    # TODO: 현재 시간 기록 (ISO 형식)
    collected_at = ________________

    all_arrivals = []

    for station in stations:
        print(f"수집 중: {station}...")
        result = api.get_arrival_info(station)

        if "realtimeArrivalList" in result:
            for arrival in result["realtimeArrivalList"]:
                # 필요한 필드만 추출
                # TODO: 딕셔너리에서 필요한 값 추출하기
                processed = {
                    "station_name": arrival.get(________________),
                    "subway_id": arrival.get("subwayId"),
                    "train_line": arrival.get("trainLineNm"),
                    "arrival_message": arrival.get("arvlMsg2"),
                    "arrival_time_sec": arrival.get("barvlDt"),
                    "received_at": arrival.get("recptnDt"),
                    "collected_at": collected_at
                }
                all_arrivals.append(processed)

    return {
        "collected_at": collected_at,
        "station_count": len(stations),
        "arrival_count": len(all_arrivals),
        "arrivals": all_arrivals
    }


def save_to_json(data: dict, output_dir: str = "data") -> str:
    """
    수집된 데이터를 JSON 파일로 저장합니다.

    Args:
        data: 저장할 데이터
        output_dir: 저장 디렉토리

    Returns:
        저장된 파일 경로
    """
    # 출력 디렉토리 생성
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # TODO: 파일명 생성 (arrivals_YYYYMMDD_HHMMSS.json)
    timestamp = datetime.now().strftime(________________)
    filename = f"arrivals_{timestamp}.json"
    filepath = Path(output_dir) / filename

    # TODO: JSON 파일로 저장 (한글 깨짐 방지: ensure_ascii=False)
    with open(filepath, "w", encoding="utf-8") as f:
        ________________

    print(f"저장 완료: {filepath}")
    return str(filepath)


if __name__ == "__main__":
    # 데이터 수집
    data = collect_arrivals()

    print(f"\n수집 결과:")
    print(f"- 수집 시간: {data['collected_at']}")
    print(f"- 역 개수: {data['station_count']}")
    print(f"- 도착 정보 개수: {data['arrival_count']}")

    # 파일 저장
    filepath = save_to_json(data)
    print(f"- 저장 파일: {filepath}")
```

<details>
<summary>💡 힌트 1: 현재 시간 ISO 형식</summary>

```python
collected_at = datetime.now().isoformat()
```

</details>

<details>
<summary>💡 힌트 2: 딕셔너리 값 추출</summary>

```python
# API 응답의 역이름 필드는 "statnNm"
"station_name": arrival.get("statnNm")
```

</details>

<details>
<summary>💡 힌트 3: 파일명 타임스탬프</summary>

```python
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
```

</details>

<details>
<summary>💡 힌트 4: JSON 저장</summary>

```python
json.dump(data, f, ensure_ascii=False, indent=2)
```

</details>

<details>
<summary>✅ 모범 답안</summary>

```python
"""
지하철 도착 정보 수집기
"""
import json
from datetime import datetime
from pathlib import Path
from api_client import SeoulMetroAPI


TARGET_STATIONS = [
    "강남", "홍대입구", "신도림", "서울역", "잠실",
    "신림", "구로디지털단지", "여의도", "종로3가", "동대문"
]


def collect_arrivals(stations: list = None) -> dict:
    if stations is None:
        stations = TARGET_STATIONS

    api = SeoulMetroAPI()
    collected_at = datetime.now().isoformat()

    all_arrivals = []

    for station in stations:
        print(f"수집 중: {station}...")
        result = api.get_arrival_info(station)

        if "realtimeArrivalList" in result:
            for arrival in result["realtimeArrivalList"]:
                processed = {
                    "station_name": arrival.get("statnNm"),
                    "subway_id": arrival.get("subwayId"),
                    "train_line": arrival.get("trainLineNm"),
                    "arrival_message": arrival.get("arvlMsg2"),
                    "arrival_time_sec": arrival.get("barvlDt"),
                    "received_at": arrival.get("recptnDt"),
                    "collected_at": collected_at
                }
                all_arrivals.append(processed)

    return {
        "collected_at": collected_at,
        "station_count": len(stations),
        "arrival_count": len(all_arrivals),
        "arrivals": all_arrivals
    }


def save_to_json(data: dict, output_dir: str = "data") -> str:
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"arrivals_{timestamp}.json"
    filepath = Path(output_dir) / filename

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"저장 완료: {filepath}")
    return str(filepath)


if __name__ == "__main__":
    data = collect_arrivals()

    print(f"\n수집 결과:")
    print(f"- 수집 시간: {data['collected_at']}")
    print(f"- 역 개수: {data['station_count']}")
    print(f"- 도착 정보 개수: {data['arrival_count']}")

    filepath = save_to_json(data)
    print(f"- 저장 파일: {filepath}")
```

</details>

---

## Step 5: 데이터 분석

### 과제 1-5: Pandas로 수집된 데이터 분석하기

`src/analyzer.py` 파일을 작성하세요.

**요구사항:**
- JSON 파일을 읽어서 Pandas DataFrame으로 변환한다
- 역별 도착 정보 개수를 집계한다
- 호선별 평균 대기 시간을 계산한다
- 결과를 CSV 파일로 저장한다

### 빈칸 채우기

```python
"""
수집된 지하철 데이터 분석기
"""
import json
import pandas as pd
from pathlib import Path


# 호선 코드 매핑
SUBWAY_LINES = {
    "1001": "1호선",
    "1002": "2호선",
    "1003": "3호선",
    "1004": "4호선",
    "1005": "5호선",
    "1006": "6호선",
    "1007": "7호선",
    "1008": "8호선",
    "1009": "9호선",
    "1063": "경의중앙선",
    "1065": "공항철도",
    "1067": "경춘선",
    "1075": "수인분당선",
    "1077": "신분당선",
}


def load_json_to_dataframe(filepath: str) -> pd.DataFrame:
    """
    JSON 파일을 읽어 DataFrame으로 변환합니다.

    Args:
        filepath: JSON 파일 경로

    Returns:
        pandas DataFrame
    """
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    # TODO: arrivals 리스트를 DataFrame으로 변환
    df = ________________

    # 호선 이름 매핑 추가
    df["line_name"] = df["subway_id"].map(SUBWAY_LINES)

    return df


def analyze_by_station(df: pd.DataFrame) -> pd.DataFrame:
    """
    역별 도착 정보 개수를 집계합니다.

    Args:
        df: 도착 정보 DataFrame

    Returns:
        역별 집계 DataFrame
    """
    # TODO: station_name으로 그룹화하여 개수 세기
    station_counts = ________________

    # 컬럼명 변경
    station_counts = station_counts.reset_index()
    station_counts.columns = ["station_name", "arrival_count"]

    # 내림차순 정렬
    station_counts = station_counts.sort_values("arrival_count", ascending=False)

    return station_counts


def analyze_by_line(df: pd.DataFrame) -> pd.DataFrame:
    """
    호선별 평균 대기 시간을 계산합니다.

    Args:
        df: 도착 정보 DataFrame

    Returns:
        호선별 집계 DataFrame
    """
    # 대기 시간을 숫자로 변환 (문자열인 경우)
    df["arrival_time_sec"] = pd.to_numeric(df["arrival_time_sec"], errors="coerce")

    # TODO: line_name으로 그룹화하여 평균 계산
    line_stats = ________________

    line_stats = line_stats.reset_index()
    line_stats.columns = ["line_name", "avg_wait_sec"]

    # 평균 대기 시간 내림차순 정렬
    line_stats = line_stats.sort_values("avg_wait_sec", ascending=False)

    return line_stats


def save_analysis_results(station_df: pd.DataFrame, line_df: pd.DataFrame,
                         output_dir: str = "output"):
    """
    분석 결과를 CSV 파일로 저장합니다.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    station_path = Path(output_dir) / "station_analysis.csv"
    line_path = Path(output_dir) / "line_analysis.csv"

    # TODO: DataFrame을 CSV로 저장 (인덱스 제외)
    ________________
    ________________

    print(f"역별 분석 저장: {station_path}")
    print(f"호선별 분석 저장: {line_path}")


if __name__ == "__main__":
    # 가장 최근 데이터 파일 찾기
    data_files = sorted(Path("data").glob("arrivals_*.json"))

    if not data_files:
        print("분석할 데이터 파일이 없습니다. 먼저 collector.py를 실행하세요.")
        exit(1)

    latest_file = data_files[-1]
    print(f"분석 대상 파일: {latest_file}")

    # 데이터 로드
    df = load_json_to_dataframe(str(latest_file))
    print(f"\n전체 데이터 개수: {len(df)}")
    print(f"컬럼: {list(df.columns)}")

    # 역별 분석
    station_analysis = analyze_by_station(df)
    print(f"\n=== 역별 도착 정보 개수 ===")
    print(station_analysis.head(10))

    # 호선별 분석
    line_analysis = analyze_by_line(df)
    print(f"\n=== 호선별 평균 대기 시간 ===")
    print(line_analysis)

    # 결과 저장
    save_analysis_results(station_analysis, line_analysis)
```

<details>
<summary>💡 힌트 1: 리스트를 DataFrame으로</summary>

```python
df = pd.DataFrame(data["arrivals"])
```

</details>

<details>
<summary>💡 힌트 2: 그룹화 후 개수 세기</summary>

```python
station_counts = df.groupby("station_name").size()
```

</details>

<details>
<summary>💡 힌트 3: 그룹화 후 평균 계산</summary>

```python
line_stats = df.groupby("line_name")["arrival_time_sec"].mean()
```

</details>

<details>
<summary>💡 힌트 4: CSV 저장</summary>

```python
station_df.to_csv(station_path, index=False, encoding="utf-8-sig")
line_df.to_csv(line_path, index=False, encoding="utf-8-sig")
```

</details>

<details>
<summary>✅ 모범 답안</summary>

```python
"""
수집된 지하철 데이터 분석기
"""
import json
import pandas as pd
from pathlib import Path


SUBWAY_LINES = {
    "1001": "1호선", "1002": "2호선", "1003": "3호선", "1004": "4호선",
    "1005": "5호선", "1006": "6호선", "1007": "7호선", "1008": "8호선",
    "1009": "9호선", "1063": "경의중앙선", "1065": "공항철도",
    "1067": "경춘선", "1075": "수인분당선", "1077": "신분당선",
}


def load_json_to_dataframe(filepath: str) -> pd.DataFrame:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data["arrivals"])
    df["line_name"] = df["subway_id"].map(SUBWAY_LINES)

    return df


def analyze_by_station(df: pd.DataFrame) -> pd.DataFrame:
    station_counts = df.groupby("station_name").size()
    station_counts = station_counts.reset_index()
    station_counts.columns = ["station_name", "arrival_count"]
    station_counts = station_counts.sort_values("arrival_count", ascending=False)

    return station_counts


def analyze_by_line(df: pd.DataFrame) -> pd.DataFrame:
    df["arrival_time_sec"] = pd.to_numeric(df["arrival_time_sec"], errors="coerce")
    line_stats = df.groupby("line_name")["arrival_time_sec"].mean()
    line_stats = line_stats.reset_index()
    line_stats.columns = ["line_name", "avg_wait_sec"]
    line_stats = line_stats.sort_values("avg_wait_sec", ascending=False)

    return line_stats


def save_analysis_results(station_df: pd.DataFrame, line_df: pd.DataFrame,
                         output_dir: str = "output"):
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    station_path = Path(output_dir) / "station_analysis.csv"
    line_path = Path(output_dir) / "line_analysis.csv"

    station_df.to_csv(station_path, index=False, encoding="utf-8-sig")
    line_df.to_csv(line_path, index=False, encoding="utf-8-sig")

    print(f"역별 분석 저장: {station_path}")
    print(f"호선별 분석 저장: {line_path}")


if __name__ == "__main__":
    data_files = sorted(Path("data").glob("arrivals_*.json"))

    if not data_files:
        print("분석할 데이터 파일이 없습니다.")
        exit(1)

    latest_file = data_files[-1]
    print(f"분석 대상 파일: {latest_file}")

    df = load_json_to_dataframe(str(latest_file))
    print(f"\n전체 데이터 개수: {len(df)}")

    station_analysis = analyze_by_station(df)
    print(f"\n=== 역별 도착 정보 개수 ===")
    print(station_analysis.head(10))

    line_analysis = analyze_by_line(df)
    print(f"\n=== 호선별 평균 대기 시간 ===")
    print(line_analysis)

    save_analysis_results(station_analysis, line_analysis)
```

</details>

---

## Step 6: 실행 및 검증

### 과제 1-6: 전체 파이프라인 실행

1. 의존성 설치
```bash
pip install requests python-dotenv pandas
```

2. 데이터 수집
```bash
cd src
python collector.py
```

3. 데이터 분석
```bash
python analyzer.py
```

### 예상 출력

<details>
<summary>📋 collector.py 예상 출력 (클릭해서 펼치기)</summary>

```
수집 중: 강남...
수집 중: 홍대입구...
수집 중: 신도림...
수집 중: 서울역...
수집 중: 잠실...
수집 중: 신림...
수집 중: 구로디지털단지...
수집 중: 여의도...
수집 중: 종로3가...
수집 중: 동대문...

수집 결과:
- 수집 시간: 2024-01-15T14:30:25.123456
- 역 개수: 10
- 도착 정보 개수: 87
저장 완료: data/arrivals_20240115_143025.json
- 저장 파일: data/arrivals_20240115_143025.json
```

</details>

<details>
<summary>📋 analyzer.py 예상 출력 (클릭해서 펼치기)</summary>

```
분석 대상 파일: data/arrivals_20240115_143025.json

전체 데이터 개수: 87
컬럼: ['station_name', 'subway_id', 'train_line', 'arrival_message',
       'arrival_time_sec', 'received_at', 'collected_at', 'line_name']

=== 역별 도착 정보 개수 ===
   station_name  arrival_count
0          강남             12
1          잠실             11
2        신도림             10
3        서울역              9
...

=== 호선별 평균 대기 시간 ===
    line_name  avg_wait_sec
0      3호선         245.5
1      2호선         198.3
2      1호선         187.2
...

역별 분석 저장: output/station_analysis.csv
호선별 분석 저장: output/line_analysis.csv
```

</details>

---

## Step 7: Git 커밋

### 과제 1-7: Phase 1 완료 커밋

```bash
cd ..  # 프로젝트 루트로 이동

# 상태 확인
git status

# 파일 추가
git add .

# 커밋 (진행자 이름 포함)
git commit -m "[Phase 1] 기본 파이프라인 구현 - Driver: 홍길동

- API 클라이언트 구현 (api_client.py)
- 데이터 수집기 구현 (collector.py)
- 데이터 분석기 구현 (analyzer.py)
- 10개 역 도착 정보 수집 및 분석 완료"

# 푸시
git push origin main
```

---

## 체크포인트

Phase 1을 완료하기 전에 다음을 확인하세요:

- [ ] API 키가 `.env`에 설정되어 있다
- [ ] `collector.py` 실행 시 `data/` 폴더에 JSON 파일이 생성된다
- [ ] `analyzer.py` 실행 시 `output/` 폴더에 CSV 파일이 생성된다
- [ ] Git 커밋이 완료되었다

---

## Phase 1의 한계 (다음 Phase에서 해결)

축하합니다! Phase 1을 완료했습니다.
하지만 이 단순한 방식에는 몇 가지 **심각한 한계**가 있습니다.

### 문제 1: 환경 의존성

**"제 컴퓨터에서는 되는데요..."**

| 항목 | 학생 A 컴퓨터 | 학생 B 컴퓨터 |
|------|---------------|---------------|
| Python | 3.11 | 3.9 |
| pandas | 2.0 | 1.5 |
| 결과 | 동작함 | 에러 발생 |

→ **Phase 2에서 Docker로 해결!**

### 문제 2: 수동 실행

**매시간 데이터를 수집하려면?**

현재: 사람이 직접 `python collector.py` 실행

| 시간 | 상태 |
|------|------|
| 08:00 | 실행 |
| 09:00 | 실행 |
| 10:00 | 깜빡함 (누락) |
| 11:00 | 실행 |
| ... | ... |

→ **Phase 5에서 GitHub Actions로 자동화!**

### 문제 3: 데이터 규모 확장

**100건 → 10,000건 → 100만 건이 되면?**

현재 코드로 100만 건 처리 시:
- 메모리 부족
- 처리 시간 수 시간
- 컴퓨터 멈춤

→ **Phase 4에서 Spark로 분산 처리!**

---

## 다음 단계

**Phase 2: Docker 컨테이너화**로 이동하세요.

> 파일: `02_Phase2_Docker컨테이너화.py`

Phase 2에서는 "제 컴퓨터에서는 되는데요" 문제를 Docker로 해결합니다.
누구의 컴퓨터에서든 동일하게 동작하는 환경을 만들어봅시다.

---

## FAQ

<details>
<summary><b>Q1. API 호출 시 에러가 발생해요</b></summary>

**체크리스트:**
1. API 키가 올바른지 확인
2. `.env` 파일이 `src/` 폴더가 아닌 프로젝트 루트에 있는지 확인
3. 인터넷 연결 확인
4. API 호출 횟수 제한(일 1000회) 초과 여부 확인

**디버깅:**
```python
import os
print(os.getenv("SEOUL_API_KEY"))  # 키가 출력되는지 확인
```

</details>

<details>
<summary><b>Q2. pandas가 설치되지 않아요</b></summary>

```bash
# pip 업그레이드 후 재설치
pip install --upgrade pip
pip install pandas
```

</details>

<details>
<summary><b>Q3. JSON 파일이 생성되지 않아요</b></summary>

1. `data/` 폴더가 존재하는지 확인
2. 쓰기 권한이 있는지 확인
3. API 응답이 비어있지 않은지 확인 (print로 디버깅)

</details>

<details>
<summary><b>Q4. 한글이 깨져서 보여요</b></summary>

파일 저장 시 인코딩 지정:
```python
# JSON
json.dump(data, f, ensure_ascii=False)

# CSV
df.to_csv(path, encoding="utf-8-sig")
```

</details>

<details>
<summary><b>Q5. "일일 호출건수 최대 1000건" 에러가 발생해요</b></summary>

**원인**: 서울 공공데이터 API는 하루 1000회 호출 제한이 있습니다.

**해결방법**:
1. 다음 날 자정(00:00)까지 기다리기 (호출 횟수 초기화)
2. 수집 간격을 늘리기 (예: 5분 → 10분)
3. 수집 대상 역 수를 줄이기 (10개 → 5개)

**에러 메시지 예시**:
```json
{
    "status": 500,
    "code": "ERROR-337",
    "message": "데이터요청은 일일 호출건수 최대 1000건을 넘을 수 없습니다."
}
```

**팁**: 개발/테스트 시에는 역 2-3개로만 테스트하고,
실제 수집 시에만 전체 역을 대상으로 하세요.

</details>

---


# Phase 2: Docker 컨테이너화

## "제 컴퓨터에서는 되는데요" 문제 해결하기

---

## 학습 목표

이 Phase를 마치면 다음을 할 수 있습니다:

- Dockerfile을 작성하여 Python 애플리케이션을 컨테이너화할 수 있다
- docker compose를 사용하여 여러 서비스를 정의하고 실행할 수 있다
- 볼륨 마운트를 통해 데이터를 영속화할 수 있다
- 환경 변수를 컨테이너에 전달할 수 있다

---

## 이 Phase의 목표

**Phase 2: Docker 컨테이너화**

**Docker Container 내부:**
- **서울시 API** → **Python (requests)** → **JSON 파일 (저장)** → **Pandas (분석)**

**호스트 볼륨 연결:** `data/`, `output/`

| 항목 | 내용 |
|------|------|
| 도구 | Docker, docker compose |
| 해결하는 문제 | 환경 의존성, "제 컴퓨터에서는 되는데요" |

---

## 역할 분담 (Phase 2)

| 역할 | 담당 |
|------|------|
| **진행자(Driver)** | 학생 B |
| **관찰자(Navigator)** | 학생 A |

> Phase 1에서 역할을 교대했습니다!

---

## 문제 상황: 환경 불일치

Phase 1에서 작성한 코드를 팀원에게 공유했더니...

**학생 A (코드 작성자):**
```bash
$ python collector.py
수집 중: 강남...
저장 완료: data/arrivals_20240115_143025.json
```

**학생 B (코드 실행자):**
```bash
$ python collector.py
ModuleNotFoundError: No module named 'dotenv'

$ pip install python-dotenv
$ python collector.py
ImportError: cannot import name 'json_normalize'
# (pandas 버전 불일치!)
```

### 왜 이런 문제가 발생하는가?

| 항목 | 학생 A | 학생 B |
|------|--------|--------|
| Python 버전 | 3.11 | 3.9 |
| pandas 버전 | 2.0.3 | 1.3.5 |
| OS | Windows | macOS |
| 설치된 패키지 | 완벽 | 일부 누락 |

> **해결책**: Docker로 **동일한 환경**을 모든 팀원에게 제공!

---

## Step 1: requirements.txt 작성

### 과제 2-1: 의존성 명시

프로젝트에 필요한 Python 패키지를 명시하는 `requirements.txt`를 작성하세요.

```bash
# requirements.txt
requests==2.31.0
python-dotenv==1.0.0
pandas==2.1.4
```

> **Tip**: 버전을 명시하면 재현 가능한 환경을 만들 수 있습니다.

---

## Step 2: Dockerfile 작성

### 과제 2-2: Python 애플리케이션용 Dockerfile 작성

`docker/Dockerfile.python` 파일을 작성하세요.

**요구사항:**
- Python 3.11 기반 이미지 사용
- 작업 디렉토리는 `/app`
- requirements.txt 설치
- 소스 코드 복사

### 빈칸 채우기

```dockerfile
# docker/Dockerfile.python

# TODO: Python 3.11 slim 이미지를 베이스로 사용
FROM ________________

# 작업 디렉토리 설정
WORKDIR /app

# TODO: requirements.txt를 컨테이너로 복사
COPY ________________ .

# TODO: pip로 의존성 설치 (캐시 사용 안 함)
RUN ________________

# 소스 코드 복사
COPY src/ ./src/

# 환경 변수 설정 (Python 출력 버퍼링 비활성화)
ENV PYTHONUNBUFFERED=1

# 기본 명령어
CMD ["python", "-m", "src.collector"]
```

<details>
<summary>💡 힌트 1: 베이스 이미지</summary>

```dockerfile
FROM python:3.11-slim
```

</details>

<details>
<summary>💡 힌트 2: 파일 복사</summary>

```dockerfile
COPY requirements.txt .
```

</details>

<details>
<summary>💡 힌트 3: pip 설치</summary>

```dockerfile
RUN pip install --no-cache-dir -r requirements.txt
```

</details>

<details>
<summary>✅ 모범 답안</summary>

```dockerfile
# docker/Dockerfile.python

# Python 3.11 slim 이미지 사용
FROM python:3.11-slim

# 작업 디렉토리 설정
WORKDIR /app

# requirements.txt 복사 및 설치
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 소스 코드 복사
COPY src/ ./src/

# 환경 변수 설정
ENV PYTHONUNBUFFERED=1

# 기본 명령어
CMD ["python", "-m", "src.collector"]
```

</details>

---

## Step 3: compose.yml 작성

### 과제 2-3: docker compose 설정 파일 작성

`docker/compose.yml` 파일을 작성하세요.

**요구사항:**
- collector 서비스: 데이터 수집
- analyzer 서비스: 데이터 분석
- 볼륨 마운트: data/, output/ 폴더
- 환경 변수: .env 파일에서 로드

### 빈칸 채우기

```yaml
# docker/compose.yml

services:
  # 데이터 수집 서비스
  collector:
    build:
      context: ..
      dockerfile: docker/Dockerfile.python
    # TODO: 볼륨 마운트 - 호스트의 data 폴더를 컨테이너의 /app/data에 연결
    volumes:
      - ________________
    # TODO: 환경 변수 파일 지정
    env_file:
      - ________________
    command: python -m src.collector

  # 데이터 분석 서비스
  analyzer:
    build:
      context: ..
      dockerfile: docker/Dockerfile.python
    volumes:
      # TODO: data 폴더 마운트 (읽기 전용)
      - ________________
      # TODO: output 폴더 마운트 (읽기/쓰기)
      - ________________
    env_file:
      - ../.env
    command: python -m src.analyzer
    # collector가 먼저 실행되어야 함
    depends_on:
      - collector

  # 대화형 셸 (디버깅용)
  shell:
    build:
      context: ..
      dockerfile: docker/Dockerfile.python
    volumes:
      - ../data:/app/data
      - ../output:/app/output
      - ../src:/app/src
    env_file:
      - ../.env
    stdin_open: true
    tty: true
    command: /bin/bash
```

<details>
<summary>💡 힌트 1: 볼륨 마운트 형식</summary>

```yaml
# 형식: 호스트경로:컨테이너경로[:옵션]
volumes:
  - ../data:/app/data
```

</details>

<details>
<summary>💡 힌트 2: 읽기 전용 마운트</summary>

```yaml
# :ro 옵션 추가
- ../data:/app/data:ro
```

</details>

<details>
<summary>💡 힌트 3: env_file 경로</summary>

```yaml
env_file:
  - ../.env  # compose.yml 기준 상대 경로
```

</details>

<details>
<summary>✅ 모범 답안</summary>

```yaml
# docker/compose.yml

services:
  collector:
    build:
      context: ..
      dockerfile: docker/Dockerfile.python
    volumes:
      - ../data:/app/data
    env_file:
      - ../.env
    command: python -m src.collector

  analyzer:
    build:
      context: ..
      dockerfile: docker/Dockerfile.python
    volumes:
      - ../data:/app/data:ro
      - ../output:/app/output
    env_file:
      - ../.env
    command: python -m src.analyzer
    depends_on:
      - collector

  shell:
    build:
      context: ..
      dockerfile: docker/Dockerfile.python
    volumes:
      - ../data:/app/data
      - ../output:/app/output
      - ../src:/app/src
    env_file:
      - ../.env
    stdin_open: true
    tty: true
    command: /bin/bash
```

</details>

---

## Step 4: 프로젝트 구조 정리

### 과제 2-4: 디렉토리 구조 확인

현재까지의 프로젝트 구조가 다음과 같은지 확인하세요.

```
seoul-metro-pipeline/
├── docker/
│   ├── compose.yml
│   └── Dockerfile.python
├── src/
│   ├── __init__.py
│   ├── api_client.py
│   ├── collector.py
│   └── analyzer.py
├── data/                    # 수집된 데이터 (Git 제외)
├── output/                  # 분석 결과 (Git 제외)
├── requirements.txt
├── .env                     # API 키 (Git 제외)
├── .env.example
├── .gitignore
└── README.md
```

### src/__init__.py 수정

패키지로 인식되도록 `src/__init__.py`를 다음과 같이 수정하세요.

```python
# src/__init__.py
"""Seoul Metro Pipeline Package"""
```

---

## Step 5: Docker로 실행

### 과제 2-5: 컨테이너 빌드 및 실행

```bash
# docker 폴더로 이동
cd docker

# 이미지 빌드
docker compose build

# 데이터 수집 실행
docker compose run --rm collector

# 데이터 분석 실행
docker compose run --rm analyzer

# (선택) 대화형 셸로 디버깅
docker compose run --rm shell
```

### 예상 출력

<details>
<summary>📋 빌드 출력 (클릭해서 펼치기)</summary>

```
[+] Building 12.5s (10/10) FINISHED
 => [internal] load build definition from Dockerfile.python
 => [internal] load .dockerignore
 => [internal] load metadata for docker.io/library/python:3.11-slim
 => [1/5] FROM docker.io/library/python:3.11-slim
 => [2/5] WORKDIR /app
 => [3/5] COPY requirements.txt .
 => [4/5] RUN pip install --no-cache-dir -r requirements.txt
 => [5/5] COPY src/ ./src/
 => exporting to image
```

</details>

<details>
<summary>📋 collector 실행 출력 (클릭해서 펼치기)</summary>

```
수집 중: 강남...
수집 중: 홍대입구...
수집 중: 신도림...
...

수집 결과:
- 수집 시간: 2024-01-15T15:30:25.123456
- 역 개수: 10
- 도착 정보 개수: 92
저장 완료: data/arrivals_20240115_153025.json
```

</details>

---

## Step 6: 볼륨과 네트워크 이해

### 개념 복습: 볼륨 마운트

**볼륨 마운트 동작 방식**

| 위치 | 경로 |
|------|------|
| 호스트 (여러분 컴퓨터) | `seoul-metro-pipeline/data/arrivals_xxx.json` |
| 컨테이너 내부 | `/app/data/arrivals_xxx.json` |

- 컨테이너에서 파일을 저장하면 → 호스트에도 저장됨
- 호스트에서 파일을 수정하면 → 컨테이너에도 반영됨
- **컨테이너를 삭제해도 데이터는 호스트에 남아있음!**

### 바인드 마운트 vs 볼륨

| 구분 | 바인드 마운트 | Docker 볼륨 |
|------|--------------|------------|
| **형식** | `./data:/app/data` | `volume_name:/app/data` |
| **위치** | 호스트 파일시스템 | Docker 관리 영역 |
| **용도** | 개발 중 코드 공유 | 프로덕션 데이터 영속화 |
| **이 과제** | ✅ 사용 | Phase 3에서 사용 |

---

## Step 7: 일반적인 문제 해결

### 과제 2-6: 트러블슈팅 연습

다음 상황에서 어떻게 해결하는지 팀원과 토론하고 해결해보세요.

#### 상황 1: 볼륨 마운트 후 파일이 안 보임

```bash
# 호스트에서는 파일이 있는데...
$ ls data/
arrivals_20240115_143025.json

# 컨테이너에서는 비어있음
$ docker compose run --rm shell ls data/
(빈 결과)
```

<details>
<summary>💡 해결 방법</summary>

**원인**: compose.yml의 경로가 잘못되었거나, 컨테이너 빌드 후 파일이 생성됨

**해결**:
```bash
# 1. 경로 확인 (compose.yml 위치 기준 상대 경로)
# compose.yml이 docker/ 폴더에 있으면:
volumes:
  - ../data:/app/data  # 상위 폴더의 data

# 2. 컨테이너 재빌드 없이 실행
docker compose run --rm shell
```

</details>

#### 상황 2: 환경 변수가 전달 안 됨

```bash
$ docker compose run --rm collector
ValueError: SEOUL_API_KEY 환경변수가 설정되지 않았습니다.
```

<details>
<summary>💡 해결 방법</summary>

**원인**: .env 파일 경로 또는 형식 문제

**해결**:
```bash
# 1. .env 파일 존재 확인
ls -la ../.env

# 2. .env 파일 형식 확인 (= 앞뒤 공백 없이)
cat ../.env
# SEOUL_API_KEY=your_key_here

# 3. compose.yml의 env_file 경로 확인
env_file:
  - ../.env  # compose.yml 기준 상대 경로
```

</details>

#### 상황 3: 포트 충돌

```bash
Error: port 5432 is already allocated
```

<details>
<summary>💡 해결 방법</summary>

**원인**: 호스트에서 같은 포트를 사용하는 프로세스가 있음

**해결**:
```bash
# 1. 사용 중인 포트 확인
lsof -i :5432

# 2. 다른 포트 사용
ports:
  - "5433:5432"  # 호스트 5433 → 컨테이너 5432

# 3. 기존 컨테이너 정리
docker compose down
docker system prune -f
```

</details>

---

## Step 8: 개발 워크플로우

### Docker 기반 개발 워크플로우

**권장 개발 워크플로우**

| 단계 | 작업 | 명령어/위치 |
|------|------|-------------|
| 1 | 코드 수정 | VSCode에서 `src/collector.py` 수정 |
| 2 | 컨테이너에서 실행 | `docker compose run --rm collector` |
| 3 | 결과 확인 | 호스트에서 `cat data/arrivals_xxx.json` |
| 4 | 문제 있으면 | 1번으로 돌아가기 |

**볼륨 마운트 덕분에 이미지 재빌드 없이 코드 수정 반영!**
(단, requirements.txt 변경 시에는 재빌드 필요)

### 유용한 명령어

```bash
# 빌드 + 실행 한번에
docker compose up --build collector

# 로그 보기
docker compose logs -f collector

# 컨테이너 내부 들어가기
docker compose run --rm shell

# 모든 컨테이너 중지 및 삭제
docker compose down

# 볼륨까지 삭제
docker compose down -v

# 이미지 삭제
docker compose down --rmi all
```

---

## Step 9: Git 커밋

### 과제 2-7: Phase 2 완료 커밋

```bash
cd ..  # 프로젝트 루트로 이동

git add .
git commit -m "[Phase 2] Docker 컨테이너화 - Driver: 김철수

- Dockerfile 작성 (Python 3.11 기반)
- docker compose 설정 (collector, analyzer, shell)
- 볼륨 마운트로 데이터 영속화
- 환경 변수 전달 설정"

git push origin main
```

---

## 체크포인트

Phase 2를 완료하기 전에 다음을 확인하세요:

- [ ] `docker compose build` 성공
- [ ] `docker compose run --rm collector` 실행 시 data/ 폴더에 JSON 생성
- [ ] `docker compose run --rm analyzer` 실행 시 output/ 폴더에 CSV 생성
- [ ] 팀원의 컴퓨터에서도 동일하게 동작
- [ ] Git 커밋 완료

---

## Phase 2의 성과와 남은 한계

### 해결된 문제

**환경 의존성 문제 해결!**

| 학생 | Before (Phase 1) | After (Phase 2) |
|------|------------------|-----------------|
| 학생 A | Python 3.11 (동작) | Docker → Python 3.11 (동작) |
| 학생 B | Python 3.9 (에러) | Docker → Python 3.11 (동작) |
| 학생 C | 패키지 없음 (에러) | Docker → Python 3.11 (동작) |

**"제 컴퓨터에서는 되는데요" → "모든 컴퓨터에서 됩니다!"**

### 남은 한계 (다음 Phase에서 해결)

#### 한계 1: 데이터 유실 위험

**현재 구조의 문제**

**API** → **Collector** → **File**

- Collector가 죽으면? → **데이터 유실!**

→ **Phase 3에서 Kafka로 해결!** (메시지 버퍼링)

#### 한계 2: 강한 결합

**API가 느리면 전체가 느려짐**

| 단계 | 소요 시간 |
|------|-----------|
| API 응답 | 5초 |
| Collector 대기 | 5초 |
| 분석 시작 | 5초 후 |

→ **Phase 3에서 Kafka로 해결!** (비동기 처리)

---

## 다음 단계

**Phase 3: Kafka 메시지 큐**로 이동하세요.

> 파일: `03_Phase3_Kafka메시지큐.py`

Phase 3에서는 데이터 유실 문제와 시스템 결합도 문제를 Kafka로 해결합니다.
Producer와 Consumer를 분리하여 더 견고한 파이프라인을 만들어봅시다.

---

## FAQ

<details>
<summary><b>Q1. docker compose build가 너무 오래 걸려요</b></summary>

**원인**: 매번 pip install을 처음부터 실행

**해결**: Docker 레이어 캐싱 활용
```dockerfile
# requirements.txt를 먼저 복사하고 설치
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 그 다음 소스 코드 복사
COPY src/ ./src/
```

이렇게 하면 requirements.txt가 변경되지 않는 한 pip install이 캐시됩니다.

</details>

<details>
<summary><b>Q2. 컨테이너에서 한글이 깨져요</b></summary>

**해결**: Dockerfile에 로케일 설정 추가
```dockerfile
ENV LANG=C.UTF-8
ENV LC_ALL=C.UTF-8
```

</details>

<details>
<summary><b>Q3. VSCode에서 컨테이너 디버깅하고 싶어요</b></summary>

**해결**: VSCode Dev Containers 확장 사용
1. Dev Containers 확장 설치
2. `Ctrl+Shift+P` → "Dev Containers: Attach to Running Container"
3. 컨테이너 선택

</details>

<details>
<summary><b>Q4. Apple Silicon (M1/M2) Mac에서 안 돼요</b></summary>

**해결**: platform 지정
```yaml
services:
  collector:
    platform: linux/amd64  # 또는 linux/arm64
    build:
      ...
```

</details>

---


# Phase 3: Kafka 메시지 큐

## "데이터 유실 없이, 시스템 간 느슨하게 연결하기"

---

## 학습 목표

이 Phase를 마치면 다음을 할 수 있습니다:

- Kafka의 핵심 개념(Topic, Producer, Consumer, Partition)을 설명할 수 있다
- confluent-kafka-python을 사용하여 메시지를 produce/consume 할 수 있다
- docker compose로 Kafka 클러스터를 실행할 수 있다
- Kafka Connect의 개념을 이해하고 간단한 설정을 할 수 있다
- 왜 메시지 큐가 필요한지 직접 체감한다

---

## 이 Phase의 목표

**Phase 3: Kafka 메시지 큐 도입**

**서울시 API** → **Producer (Python)** → **Kafka Topic** → **다중 Consumer**

| Consumer | 역할 |
|----------|------|
| Consumer A | 저장 |
| Consumer B | 분석 |
| Consumer C | 알림 |

| 항목 | 내용 |
|------|------|
| 도구 | Apache Kafka, confluent-kafka-python |
| 데이터 규모 | ~10,000건 |
| 해결하는 문제 | 데이터 유실, 시스템 결합도, 확장성 |

---

## 역할 분담 (Phase 3)

| 역할 | 담당 |
|------|------|
| **진행자(Driver)** | 학생 A |
| **관찰자(Navigator)** | 학생 B |

> Phase 2에서 역할을 교대했습니다!

---

## 문제 상황: 왜 Kafka가 필요한가?

Phase 2까지의 구조를 살펴봅시다:

**현재 구조의 문제점들**

**문제 1: 데이터 유실**
- **API** → **Collector** → **File**
- Collector가 죽으면? → 받던 데이터 유실!

**문제 2: 강한 결합 (Tight Coupling)**
- **API 느림** → **Collector 대기** → **분석도 대기**
- API가 5초 걸리면 전체가 5초 지연

**문제 3: 확장성 부족**
- Consumer를 추가하려면? 코드 수정 필요
- **Collector** → **File** AND **DB** AND **알림**... (if문 지옥)

### Kafka로 해결!

**Kafka 도입 후**

**API** → **Producer** → **Kafka Topic** → 다중 Consumer

| Kafka Topic 연결 | 역할 |
|------------------|------|
| Consumer A | 파일 저장 |
| Consumer B | DB 저장 |
| Consumer C | 알림 발송 |

- Producer가 죽어도 Kafka에 데이터 보존
- Consumer 추가/삭제가 자유로움
- 각 컴포넌트가 독립적으로 동작

---

## Kafka 핵심 개념

### 1. 기본 구성요소

**Kafka 아키텍처**

```
Producer → Kafka Cluster → Consumer Group
               ↓
        Topic: metro-arrivals
               ↓
        Partition 0, 1, 2...
```

| 구성요소 | 역할 |
|----------|------|
| Kafka Cluster | 메시지 브로커 |
| Topic | 메시지 분류 단위 (예: metro-arrivals) |
| Partition | Topic 내 병렬 처리 단위 |
| Producer | 메시지 발행 |
| Consumer Group | 메시지 소비 (Consumer 1, 2...) |

### 2. 용어 정리

| 용어 | 설명 | 비유 |
|------|------|------|
| **Topic** | 메시지를 분류하는 카테고리 | 게시판 |
| **Partition** | Topic을 나눈 단위, 병렬 처리 | 게시판의 페이지 |
| **Producer** | 메시지를 보내는 주체 | 글 작성자 |
| **Consumer** | 메시지를 읽는 주체 | 글 구독자 |
| **Consumer Group** | Consumer들의 논리적 그룹 | 구독자 모임 |
| **Offset** | 파티션 내 메시지 위치 | 페이지 번호 |
| **Broker** | Kafka 서버 | 게시판 서버 |

### 3. 왜 분산 환경에서 강력한가?

**단일 브로커 vs 멀티 브로커**

| 환경 | 브로커 구성 | 용도 |
|------|------------|------|
| 이 과제 | Broker 1 (단일) | 학습용 |
| 실무 | Broker 1 (Leader) + Broker 2, 3 (Replica) | 고가용성 |

**멀티 브로커의 장점:**
- Broker 1이 죽어도 Broker 2가 대신 처리
- 데이터 복제로 유실 방지

> 이 과제에서는 단일 브로커로 개념을 익히고, 실무에서는 멀티 브로커로 고가용성을 확보합니다.

---

## Step 1: Kafka 환경 구성

### 과제 3-1: docker compose에 Kafka 추가

`docker/compose.yml` 파일을 수정하여 Kafka를 추가하세요.

**요구사항:**
- Kafka 브로커 (KRaft 모드 - Zookeeper 없이 동작)
- kafka-ui (웹 UI로 Kafka 모니터링)

> **KRaft 모드란?**
> Kafka 3.3부터 정식 지원되는 Zookeeper 없는 운영 모드입니다.
> Kafka 자체적으로 Raft 합의 알고리즘을 사용하여 메타데이터를 관리합니다.
> 운영 복잡도가 줄어들고, 시작 시간이 빨라지는 장점이 있습니다.

### compose.yml 수정

```yaml
# docker/compose.yml

services:
  # ============================================
  # Kafka 인프라 (KRaft 모드 - Zookeeper 불필요)
  # ============================================

  kafka:
    image: apache/kafka:4.1.1
    container_name: metro-kafka
    hostname: kafka
    ports:
      - "9092:9092"
    environment:
      # KRaft 모드 설정
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:9093
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_INTER_BROKER_LISTENER_NAME: PLAINTEXT
      # 단일 브로커 환경 설정
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
      KAFKA_LOG_DIRS: /var/lib/kafka/data
      CLUSTER_ID: metro-pipeline-cluster-001
    volumes:
      - kafka-data:/var/lib/kafka/data
    healthcheck:
      test: ["CMD-SHELL", "/opt/kafka/bin/kafka-broker-api-versions.sh --bootstrap-server localhost:9092 || exit 1"]
      interval: 10s
      timeout: 10s
      retries: 5
      start_period: 30s

  # Kafka 웹 UI (모니터링용)
  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    container_name: metro-kafka-ui
    depends_on:
      kafka:
        condition: service_healthy
    ports:
      - "8080:8080"
    environment:
      KAFKA_CLUSTERS_0_NAME: metro-cluster
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: kafka:9092

  # ============================================
  # 애플리케이션 서비스
  # ============================================

  producer:
    build:
      context: ..
      dockerfile: docker/Dockerfile.python
    depends_on:
      kafka:
        condition: service_healthy
    volumes:
      - ../src:/app
    env_file:
      - ../.env
    environment:
      KAFKA_BOOTSTRAP_SERVERS: kafka:9092
    working_dir: /app
    command: python src/producer.py

  consumer:
    build:
      context: ..
      dockerfile: docker/Dockerfile.python
    depends_on:
      kafka:
        condition: service_healthy
    volumes:
      - ../src:/app
      - ../data:/data
    env_file:
      - ../.env
    environment:
      KAFKA_BOOTSTRAP_SERVERS: kafka:9092
    working_dir: /app
    command: python src/consumer.py

volumes:
  kafka-data:
```

---

## Step 2: requirements.txt 업데이트

### 과제 3-2: Kafka 라이브러리 추가

```bash
# requirements.txt
requests==2.31.0
python-dotenv==1.0.0
pandas==2.1.4
confluent-kafka==2.3.0
```

---

## Step 3: Kafka 인프라 시작

### 과제 3-3: Kafka 클러스터 시작 및 확인

```bash
cd docker

# Kafka 인프라 시작 (백그라운드)
# KRaft 모드는 Zookeeper가 필요 없습니다!
docker compose up -d kafka kafka-ui

# 상태 확인
docker compose ps

# 로그 확인 (Kafka가 정상 시작되었는지)
docker compose logs kafka | tail -20
```

### 예상 출력

<details>
<summary>📋 정상 시작 시 출력 (클릭해서 펼치기)</summary>

```
NAME             IMAGE                           STATUS
metro-kafka      apache/kafka:4.1.1              Up 30 seconds (healthy)
metro-kafka-ui   provectuslabs/kafka-ui:latest   Up 10 seconds
```

Kafka 로그에서 확인할 내용:
```
[KafkaRaftServer nodeId=1] Kafka Server started
```

> **참고**: KRaft 모드에서는 `KafkaRaftServer`로 시작됩니다.
> 기존 Zookeeper 모드의 `KafkaServer`와 다릅니다.

</details>

### Kafka UI 확인

브라우저에서 http://localhost:8080 접속하여 Kafka UI를 확인하세요.

**Kafka UI (http://localhost:8080) 예상 화면:**

| 항목 | 값 |
|------|-----|
| Clusters | local |
| Brokers | 1 |
| Topics | 0 (아직 생성 안 됨) |

---

## Step 4: Producer 구현

### 과제 3-4: Kafka Producer 작성

`src/kafka_producer.py` 파일을 작성하세요.

**요구사항:**
- API에서 데이터를 가져와 Kafka로 전송
- 메시지는 JSON 형식
- 토픽 이름: `metro-arrivals`

### 빈칸 채우기

```python
"""
Kafka Producer - 지하철 도착 정보를 Kafka로 전송
"""
import os
import json
import time
from datetime import datetime
from confluent_kafka import Producer
from api_client import SeoulMetroAPI


# 수집할 역 목록
TARGET_STATIONS = [
    "강남", "홍대입구", "신도림", "서울역", "잠실",
    "신림", "구로디지털단지", "여의도", "종로3가", "동대문"
]

# Kafka 설정
KAFKA_TOPIC = "metro-arrivals"


def create_producer() -> Producer:
    """Kafka Producer 생성"""
    # TODO: 환경변수에서 Kafka 서버 주소 가져오기
    bootstrap_servers = os.getenv("KAFKA_BOOTSTRAP_SERVERS", ________________)

    config = {
        "bootstrap.servers": bootstrap_servers,
        "client.id": "metro-producer",
    }

    return Producer(config)


def delivery_callback(err, msg):
    """메시지 전송 결과 콜백"""
    if err:
        print(f"❌ 전송 실패: {err}")
    else:
        print(f"✅ 전송 성공: {msg.topic()} [{msg.partition()}] @ {msg.offset()}")


def produce_arrivals(producer: Producer, stations: list = None):
    """
    역 도착 정보를 Kafka로 전송
    """
    if stations is None:
        stations = TARGET_STATIONS

    api = SeoulMetroAPI()
    collected_at = datetime.now().isoformat()
    message_count = 0

    for station in stations:
        print(f"수집 중: {station}...")
        result = api.get_arrival_info(station)

        if "realtimeArrivalList" not in result:
            continue

        for arrival in result["realtimeArrivalList"]:
            # 메시지 생성
            message = {
                "station_name": arrival.get("statnNm"),
                "subway_id": arrival.get("subwayId"),
                "train_line": arrival.get("trainLineNm"),
                "arrival_message": arrival.get("arvlMsg2"),
                "arrival_time_sec": arrival.get("barvlDt"),
                "received_at": arrival.get("recptnDt"),
                "collected_at": collected_at
            }

            # TODO: 메시지를 JSON으로 직렬화하여 Kafka로 전송
            # producer.produce(토픽, 값, 콜백)
            producer.produce(
                ________________,
                ________________,
                callback=delivery_callback
            )

            message_count += 1

        # 배치 전송 (버퍼 비우기)
        producer.poll(0)

    # 모든 메시지 전송 완료 대기
    # TODO: 버퍼에 남은 메시지 모두 전송
    ________________

    return message_count


def run_continuous(interval_seconds: int = 60):
    """
    지속적으로 데이터 수집 및 전송 (스트림 처리 시뮬레이션)
    """
    producer = create_producer()
    print(f"Producer 시작 - {interval_seconds}초 간격으로 수집")

    try:
        while True:
            print(f"\n[{datetime.now().isoformat()}] 수집 시작...")
            count = produce_arrivals(producer)
            print(f"전송 완료: {count}개 메시지")

            print(f"{interval_seconds}초 대기...")
            time.sleep(interval_seconds)

    except KeyboardInterrupt:
        print("\n종료 요청...")
    finally:
        producer.flush()
        print("Producer 종료")


if __name__ == "__main__":
    # 단일 실행 모드
    producer = create_producer()
    count = produce_arrivals(producer)
    print(f"\n총 {count}개 메시지 전송 완료")

    # 지속 실행 모드 (주석 해제하여 사용)
    # run_continuous(interval_seconds=60)
```

<details>
<summary>💡 힌트 1: 기본 Kafka 서버 주소</summary>

```python
# 컨테이너 내부에서는 kafka:29092, 외부에서는 localhost:9092
bootstrap_servers = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
```

</details>

<details>
<summary>💡 힌트 2: 메시지 전송</summary>

```python
producer.produce(
    KAFKA_TOPIC,
    json.dumps(message).encode("utf-8"),
    callback=delivery_callback
)
```

</details>

<details>
<summary>💡 힌트 3: 버퍼 비우기</summary>

```python
producer.flush()
```

</details>

<details>
<summary>✅ 모범 답안</summary>

```python
"""
Kafka Producer - 지하철 도착 정보를 Kafka로 전송
"""
import os
import json
import time
from datetime import datetime
from confluent_kafka import Producer
from api_client import SeoulMetroAPI


TARGET_STATIONS = [
    "강남", "홍대입구", "신도림", "서울역", "잠실",
    "신림", "구로디지털단지", "여의도", "종로3가", "동대문"
]

KAFKA_TOPIC = "metro-arrivals"


def create_producer() -> Producer:
    bootstrap_servers = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")

    config = {
        "bootstrap.servers": bootstrap_servers,
        "client.id": "metro-producer",
    }

    return Producer(config)


def delivery_callback(err, msg):
    if err:
        print(f"❌ 전송 실패: {err}")
    else:
        print(f"✅ 전송 성공: {msg.topic()} [{msg.partition()}] @ {msg.offset()}")


def produce_arrivals(producer: Producer, stations: list = None):
    if stations is None:
        stations = TARGET_STATIONS

    api = SeoulMetroAPI()
    collected_at = datetime.now().isoformat()
    message_count = 0

    for station in stations:
        print(f"수집 중: {station}...")
        result = api.get_arrival_info(station)

        if "realtimeArrivalList" not in result:
            continue

        for arrival in result["realtimeArrivalList"]:
            message = {
                "station_name": arrival.get("statnNm"),
                "subway_id": arrival.get("subwayId"),
                "train_line": arrival.get("trainLineNm"),
                "arrival_message": arrival.get("arvlMsg2"),
                "arrival_time_sec": arrival.get("barvlDt"),
                "received_at": arrival.get("recptnDt"),
                "collected_at": collected_at
            }

            producer.produce(
                KAFKA_TOPIC,
                json.dumps(message).encode("utf-8"),
                callback=delivery_callback
            )

            message_count += 1

        producer.poll(0)

    producer.flush()

    return message_count


def run_continuous(interval_seconds: int = 60):
    producer = create_producer()
    print(f"Producer 시작 - {interval_seconds}초 간격으로 수집")

    try:
        while True:
            print(f"\n[{datetime.now().isoformat()}] 수집 시작...")
            count = produce_arrivals(producer)
            print(f"전송 완료: {count}개 메시지")

            print(f"{interval_seconds}초 대기...")
            time.sleep(interval_seconds)

    except KeyboardInterrupt:
        print("\n종료 요청...")
    finally:
        producer.flush()
        print("Producer 종료")


if __name__ == "__main__":
    producer = create_producer()
    count = produce_arrivals(producer)
    print(f"\n총 {count}개 메시지 전송 완료")
```

</details>

---

## Step 5: Consumer 구현

### 과제 3-5: Kafka Consumer 작성

`src/kafka_consumer.py` 파일을 작성하세요.

**요구사항:**
- `metro-arrivals` 토픽 구독
- 메시지를 JSON으로 파싱
- 파일로 저장 (배치 단위)

### 빈칸 채우기

```python
"""
Kafka Consumer - 지하철 도착 정보를 소비하여 저장
"""
import os
import json
from datetime import datetime
from pathlib import Path
from confluent_kafka import Consumer, KafkaException


KAFKA_TOPIC = "metro-arrivals"


def create_consumer() -> Consumer:
    """Kafka Consumer 생성"""
    bootstrap_servers = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")

    config = {
        "bootstrap.servers": bootstrap_servers,
        # TODO: Consumer Group ID 설정
        "group.id": ________________,
        # 가장 처음부터 읽기
        "auto.offset.reset": "earliest",
        # 수동 커밋 (처리 완료 후 커밋)
        "enable.auto.commit": False,
    }

    return Consumer(config)


def consume_and_save(consumer: Consumer, batch_size: int = 100,
                     output_dir: str = "data"):
    """
    메시지를 소비하고 파일로 저장
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # TODO: 토픽 구독
    consumer.________________([KAFKA_TOPIC])

    print(f"Consumer 시작 - 토픽: {KAFKA_TOPIC}")
    print(f"배치 크기: {batch_size}")

    messages = []
    message_count = 0

    try:
        while True:
            # TODO: 메시지 폴링 (1초 타임아웃)
            msg = consumer.________________(timeout=1.0)

            if msg is None:
                # 타임아웃 - 새 메시지 없음
                if messages:
                    # 대기 중인 메시지가 있으면 저장
                    save_batch(messages, output_dir)
                    # TODO: 오프셋 커밋
                    consumer.________________()
                    messages = []
                continue

            if msg.error():
                raise KafkaException(msg.error())

            # 메시지 파싱
            value = json.loads(msg.value().decode("utf-8"))
            messages.append(value)
            message_count += 1

            print(f"수신: {value.get('station_name')} - {value.get('arrival_message')}")

            # 배치 크기 도달 시 저장
            if len(messages) >= batch_size:
                save_batch(messages, output_dir)
                consumer.commit()
                print(f"배치 저장 완료: {len(messages)}개")
                messages = []

    except KeyboardInterrupt:
        print("\n종료 요청...")
    finally:
        # 남은 메시지 저장
        if messages:
            save_batch(messages, output_dir)
            consumer.commit()

        consumer.close()
        print(f"Consumer 종료 - 총 {message_count}개 메시지 처리")


def save_batch(messages: list, output_dir: str):
    """배치 단위로 파일 저장"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"kafka_batch_{timestamp}.json"
    filepath = Path(output_dir) / filename

    data = {
        "saved_at": datetime.now().isoformat(),
        "message_count": len(messages),
        "messages": messages
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"저장: {filepath}")


if __name__ == "__main__":
    consumer = create_consumer()
    consume_and_save(consumer, batch_size=50)
```

<details>
<summary>💡 힌트 1: Consumer Group ID</summary>

```python
"group.id": "metro-consumer-group"
```

</details>

<details>
<summary>💡 힌트 2: 토픽 구독</summary>

```python
consumer.subscribe([KAFKA_TOPIC])
```

</details>

<details>
<summary>💡 힌트 3: 메시지 폴링</summary>

```python
msg = consumer.poll(timeout=1.0)
```

</details>

<details>
<summary>💡 힌트 4: 오프셋 커밋</summary>

```python
consumer.commit()
```

</details>

<details>
<summary>✅ 모범 답안</summary>

```python
"""
Kafka Consumer - 지하철 도착 정보를 소비하여 저장
"""
import os
import json
from datetime import datetime
from pathlib import Path
from confluent_kafka import Consumer, KafkaException


KAFKA_TOPIC = "metro-arrivals"


def create_consumer() -> Consumer:
    bootstrap_servers = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")

    config = {
        "bootstrap.servers": bootstrap_servers,
        "group.id": "metro-consumer-group",
        "auto.offset.reset": "earliest",
        "enable.auto.commit": False,
    }

    return Consumer(config)


def consume_and_save(consumer: Consumer, batch_size: int = 100,
                     output_dir: str = "data"):
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    consumer.subscribe([KAFKA_TOPIC])

    print(f"Consumer 시작 - 토픽: {KAFKA_TOPIC}")
    print(f"배치 크기: {batch_size}")

    messages = []
    message_count = 0

    try:
        while True:
            msg = consumer.poll(timeout=1.0)

            if msg is None:
                if messages:
                    save_batch(messages, output_dir)
                    consumer.commit()
                    messages = []
                continue

            if msg.error():
                raise KafkaException(msg.error())

            value = json.loads(msg.value().decode("utf-8"))
            messages.append(value)
            message_count += 1

            print(f"수신: {value.get('station_name')} - {value.get('arrival_message')}")

            if len(messages) >= batch_size:
                save_batch(messages, output_dir)
                consumer.commit()
                print(f"배치 저장 완료: {len(messages)}개")
                messages = []

    except KeyboardInterrupt:
        print("\n종료 요청...")
    finally:
        if messages:
            save_batch(messages, output_dir)
            consumer.commit()

        consumer.close()
        print(f"Consumer 종료 - 총 {message_count}개 메시지 처리")


def save_batch(messages: list, output_dir: str):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"kafka_batch_{timestamp}.json"
    filepath = Path(output_dir) / filename

    data = {
        "saved_at": datetime.now().isoformat(),
        "message_count": len(messages),
        "messages": messages
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"저장: {filepath}")


if __name__ == "__main__":
    consumer = create_consumer()
    consume_and_save(consumer, batch_size=50)
```

</details>

---

## Step 6: Producer/Consumer 실행

### 과제 3-6: Kafka 파이프라인 실행

터미널 2개를 열어서 실행합니다.

#### 터미널 1: Consumer 먼저 실행

```bash
cd docker
docker compose run --rm consumer
```

#### 터미널 2: Producer 실행

```bash
cd docker
docker compose run --rm producer
```

### 예상 출력

<details>
<summary>📋 Producer 출력 (클릭해서 펼치기)</summary>

```
수집 중: 강남...
✅ 전송 성공: metro-arrivals [0] @ 0
✅ 전송 성공: metro-arrivals [0] @ 1
✅ 전송 성공: metro-arrivals [0] @ 2
...
수집 중: 홍대입구...
✅ 전송 성공: metro-arrivals [0] @ 12
...

총 87개 메시지 전송 완료
```

</details>

<details>
<summary>📋 Consumer 출력 (클릭해서 펼치기)</summary>

```
Consumer 시작 - 토픽: metro-arrivals
배치 크기: 50
수신: 강남 - 전역 도착
수신: 강남 - 2분 후 도착
...
수신: 홍대입구 - 잠시 후 도착
배치 저장 완료: 50개
저장: data/kafka_batch_20240115_160030.json
...
```

</details>

### Kafka UI에서 확인

http://localhost:8080 에서:
- Topics → `metro-arrivals` 클릭
- Messages 탭에서 전송된 메시지 확인

---

## Step 7: Kafka Connect 소개 (보너스)

### Kafka Connect란?

Kafka Connect는 **코드 없이** 외부 시스템과 Kafka를 연결하는 프레임워크입니다.

**Kafka Connect 비교**

| 항목 | 직접 코드 작성 | Kafka Connect 사용 |
|------|----------------|-------------------|
| 설정 | Python Producer 작성 | JSON 설정 파일만 작성 |
| 에러 처리 | 직접 구현 | 내장 |
| 재시작 | 직접 구현 | 자동 |

**Source Connector:** Source (DB) → Connect (Source Conn) → Kafka

**Sink Connector:** Kafka → Connect (Sink Conn) → Sink (S3)

### Connector 종류

| 종류 | 방향 | 예시 |
|------|------|------|
| **Source Connector** | 외부 → Kafka | DB → Kafka, File → Kafka |
| **Sink Connector** | Kafka → 외부 | Kafka → S3, Kafka → ES |

### 실습: FileStream Sink Connector (선택)

Kafka의 메시지를 파일로 저장하는 간단한 Connector를 설정해봅니다.

<details>
<summary>📋 FileStream Sink Connector 설정 (클릭해서 펼치기)</summary>

compose.yml에 Kafka Connect 추가:

```yaml
kafka-connect:
  image: confluentinc/cp-kafka-connect:7.5.0
  depends_on:
    - kafka
  ports:
    - "8083:8083"
  environment:
    CONNECT_BOOTSTRAP_SERVERS: kafka:29092
    CONNECT_REST_PORT: 8083
    CONNECT_GROUP_ID: "connect-cluster"
    CONNECT_CONFIG_STORAGE_TOPIC: "connect-configs"
    CONNECT_OFFSET_STORAGE_TOPIC: "connect-offsets"
    CONNECT_STATUS_STORAGE_TOPIC: "connect-status"
    CONNECT_CONFIG_STORAGE_REPLICATION_FACTOR: 1
    CONNECT_OFFSET_STORAGE_REPLICATION_FACTOR: 1
    CONNECT_STATUS_STORAGE_REPLICATION_FACTOR: 1
    CONNECT_KEY_CONVERTER: "org.apache.kafka.connect.storage.StringConverter"
    CONNECT_VALUE_CONVERTER: "org.apache.kafka.connect.json.JsonConverter"
    CONNECT_VALUE_CONVERTER_SCHEMAS_ENABLE: "false"
    CONNECT_PLUGIN_PATH: "/usr/share/java"
  volumes:
    - ../data/kafka-connect:/data
```

Connector 생성 API 호출:

```bash
curl -X POST http://localhost:8083/connectors \
  -H "Content-Type: application/json" \
  -d '{
    "name": "file-sink",
    "config": {
      "connector.class": "org.apache.kafka.connect.file.FileStreamSinkConnector",
      "tasks.max": "1",
      "topics": "metro-arrivals",
      "file": "/data/metro-arrivals.txt"
    }
  }'
```

Connector 상태 확인:

```bash
curl http://localhost:8083/connectors/file-sink/status
```

</details>

> **참고**: Kafka Connect는 실무에서 매우 유용하지만, 이 과제에서는 개념 이해 수준으로만 다룹니다.
> 다음 수업에서 Elasticsearch Sink Connector를 사용할 때 더 자세히 배웁니다.

---

## Step 8: Git 커밋

### 과제 3-7: Phase 3 완료 커밋

```bash
cd ..  # 프로젝트 루트로 이동

git add .
git commit -m "[Phase 3] Kafka 메시지 큐 도입 - Driver: 홍길동

- Kafka (KRaft 모드), kafka-ui docker compose 설정
- Kafka Producer 구현 (confluent-kafka-python)
- Kafka Consumer 구현 (배치 저장)
- 토픽: metro-arrivals"

git push origin main
```

---

## 체크포인트

Phase 3을 완료하기 전에 다음을 확인하세요:

- [ ] `docker compose up -d kafka kafka-ui` 성공
- [ ] http://localhost:8080 에서 Kafka UI 접속 가능
- [ ] Producer 실행 시 메시지 전송 성공 로그 출력
- [ ] Consumer 실행 시 메시지 수신 및 파일 저장
- [ ] Kafka UI에서 `metro-arrivals` 토픽과 메시지 확인
- [ ] Git 커밋 완료

---

## Phase 3의 성과와 남은 한계

### 해결된 문제

**데이터 유실 방지**

| 시점 | 흐름 | 결과 |
|------|------|------|
| Before | API → Collector → (죽음) | 유실 |
| After | API → Producer → Kafka → Consumer | 보존 |

Producer가 죽어도 Kafka에 데이터 보존!

**느슨한 결합 (Loose Coupling)**

| 시점 | 흐름 |
|------|------|
| Before | API 느림 → Collector 대기 → 분석도 대기 |
| After | API 느림 → Producer → Kafka → Consumer 독립 동작 |

**확장성**

Consumer 추가가 자유로움:
- Kafka → Consumer A: 파일
- Kafka → Consumer B: DB (쉽게 추가!)
- Kafka → Consumer C: 알림 (쉽게 추가!)

### 남은 한계 (다음 Phase에서 해결)

**데이터가 10,000건 → 100만 건이 되면?**

현재 Consumer의 한계:
- 단일 스레드로 처리
- 메모리에 모든 데이터 로드
- 복잡한 집계 연산이 느림

100만 건 처리 시:
- 메모리 부족
- 처리 시간 수 시간

→ **Phase 4에서 Spark로 분산 처리!**

---

## 다음 단계

**Phase 4: Spark 분산 처리**로 이동하세요.

> 파일: `04_Phase4_Spark분산처리.py`

Phase 4에서는 대용량 데이터 처리의 한계를 Spark로 해결합니다.
Kafka에서 데이터를 읽어 Spark Structured Streaming으로 처리해봅시다.

---

## FAQ

<details>
<summary><b>Q1. Kafka가 시작되지 않아요</b></summary>

**체크리스트:**
1. 포트 충돌 확인 (9092)
2. Docker 리소스 확인 (메모리 최소 2GB)
3. healthcheck가 통과할 때까지 30초 정도 대기

> **참고**: KRaft 모드는 Zookeeper가 필요 없어 시작이 더 간단합니다.

```bash
# 로그 확인
docker compose logs kafka

# 재시작
docker compose down
docker compose up -d kafka kafka-ui
```

</details>

<details>
<summary><b>Q2. Producer가 Kafka에 연결되지 않아요</b></summary>

**원인**: 호스트명/포트 문제

**해결**:
- 컨테이너 내부: `kafka:9092`
- 호스트에서 직접 접속은 지원하지 않음 (컨테이너 환경에서만 사용)

```python
# 환경변수 확인
print(os.getenv("KAFKA_BOOTSTRAP_SERVERS"))
# 출력: kafka:9092
```

</details>

<details>
<summary><b>Q3. Consumer가 메시지를 못 받아요</b></summary>

**체크리스트:**
1. 토픽 이름이 일치하는지 확인
2. `auto.offset.reset` 설정 확인
3. Consumer Group이 이미 메시지를 읽었을 수 있음

```bash
# 새 Consumer Group으로 처음부터 읽기
"group.id": "new-consumer-group"
"auto.offset.reset": "earliest"
```

</details>

<details>
<summary><b>Q4. Kafka UI에 토픽이 안 보여요</b></summary>

Producer가 최소 한 번 실행되어야 토픽이 생성됩니다.
(auto.create.topics.enable=true 설정)

```bash
# Producer 실행
docker compose run --rm producer

# UI 새로고침
```

</details>

---


# Phase 4: Spark 분산 처리

## "100만 건도 거뜬히 처리하기"

---

## 학습 목표

이 Phase를 마치면 다음을 할 수 있습니다:

- Spark의 핵심 개념(SparkSession, DataFrame, Transformation/Action)을 이해한다
- Spark Structured Streaming으로 Kafka 데이터를 실시간 처리할 수 있다
- DataFrame API로 데이터 변환, 집계, 윈도우 연산을 수행할 수 있다
- 왜 분산 처리가 필요한지 직접 체감한다

---

## 이 Phase의 목표

**Phase 4: Spark 분산 처리**

**Kafka Topic** → **Spark Cluster (Structured Streaming)** → **출력**

| Spark Cluster 구성 | 역할 |
|--------------------|------|
| Worker 1, 2, 3 | 분산 처리 |

| 출력 대상 | 용도 |
|-----------|------|
| Console | 테스트 |
| File (Parquet) | 영구 저장 |

| 항목 | 내용 |
|------|------|
| 도구 | Apache Spark, PySpark, Structured Streaming |
| 데이터 규모 | ~1,000,000건 |
| 해결하는 문제 | 대용량 처리 성능, 복잡한 집계 연산 |

---

## 역할 분담 (Phase 4)

| 역할 | 담당 |
|------|------|
| **진행자(Driver)** | 학생 B |
| **관찰자(Navigator)** | 학생 A |

> Phase 3에서 역할을 교대했습니다!

---

## 문제 상황: 대용량 데이터 처리의 한계

Phase 3의 Python Consumer로 100만 건을 처리하면 어떻게 될까요?

**Python Consumer의 한계**

| 데이터 양 | 처리 시간 | 메모리 사용 | 결과 |
|-----------|-----------|-------------|------|
| 100건 | 1초 | 10MB | OK |
| 10,000건 | 30초 | 200MB | OK |
| 100,000건 | 5분 | 2GB | 느림 |
| 1,000,000건 | 1시간+ | 8GB+ | 메모리 부족! |

**문제점:**
1. 단일 스레드 처리 → CPU 1개만 사용
2. 전체 데이터를 메모리에 로드 → 메모리 한계
3. pandas 집계 연산 → 대용량에서 매우 느림

### Spark로 해결!

**Spark의 분산 처리**

**1,000,000건 데이터** → 분산 → **Worker 1, 2, 3** → 병합 → **결과**

| Worker | 처리 데이터 |
|--------|-------------|
| Worker 1 | 333,333건 |
| Worker 2 | 333,333건 |
| Worker 3 | 333,334건 |

**장점:**
- 병렬 처리 → 3배 빠름 (Worker 수에 비례)
- 분산 메모리 → 각 Worker가 일부만 처리
- 최적화된 연산 → Catalyst Optimizer

---

## Spark 핵심 개념

### 1. SparkSession

```python
# SparkSession: Spark 애플리케이션의 진입점
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MetroPipeline") \
    .master("local[*]") \  # 로컬 모드, 모든 코어 사용
    .getOrCreate()
```

### 2. DataFrame

**Spark DataFrame vs Pandas DataFrame**

| 항목 | Pandas DataFrame | Spark DataFrame |
|------|------------------|-----------------|
| 저장 | 단일 노드 메모리 | 클러스터 분산 저장 |
| 실행 | 즉시 실행 | 지연 실행 (Lazy Evaluation) |
| 기반 | Python 객체 | JVM 기반 최적화 |

| 연산 예시 | Pandas | Spark |
|-----------|--------|-------|
| 데이터 확인 | `df.head()` → 즉시 실행 | `df.show()` → Action 시점에 실행 |
| 그룹화 | `df.groupby()` → 즉시 실행 | `df.groupBy()` → Transformation (지연) |

### 3. Transformation vs Action

| 구분 | 설명 | 예시 | 실행 시점 |
|------|------|------|----------|
| **Transformation** | 새 DataFrame 생성 | filter, select, groupBy | 지연 (Lazy) |
| **Action** | 결과 반환/저장 | show, collect, write | 즉시 실행 |

```python
# Transformation (지연 실행 - 계획만 세움)
df2 = df.filter(df.station == "강남")
df3 = df2.groupBy("line_name").count()

# Action (여기서 실제 실행!)
df3.show()  # 이 시점에 위의 모든 연산이 한꺼번에 실행됨
```

### 4. 왜 분산 환경에서 강력한가?

**단일 노드 vs 멀티 노드 클러스터**

| 환경 | 구성 | 설명 |
|------|------|------|
| 이 과제 (학습용) | Spark Local (모든 코어) | `local[*]` 모드: 단일 머신의 모든 코어 사용 |
| 실무 (멀티 노드) | Driver (조율자) + Worker 1, 2... (실행자) | 클러스터 분산 처리 |

> 이 과제에서는 단일 노드로 개념을 익히고, 실무에서는 클러스터로 확장합니다.
> 코드는 거의 동일 - master 설정만 변경!

---

## Step 1: Spark 환경 구성

### 과제 4-1: docker compose에 Spark 추가

`docker/Dockerfile.spark` 파일을 생성하세요.

```dockerfile
# docker/Dockerfile.spark

FROM apache/spark:3.5.8

USER root

# 필요한 Python 패키지 설치
RUN pip install --no-cache-dir \
    pandas \
    pyarrow

# Kafka 연동을 위한 JAR 파일 다운로드
RUN curl -L -o /opt/spark/jars/spark-sql-kafka-0-10_2.12-3.5.0.jar \
    https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar && \
    curl -L -o /opt/spark/jars/kafka-clients-3.5.0.jar \
    https://repo1.maven.org/maven2/org/apache/kafka/kafka-clients/3.5.0/kafka-clients-3.5.0.jar && \
    curl -L -o /opt/spark/jars/spark-token-provider-kafka-0-10_2.12-3.5.0.jar \
    https://repo1.maven.org/maven2/org/apache/spark/spark-token-provider-kafka-0-10_2.12/3.5.0/spark-token-provider-kafka-0-10_2.12-3.5.0.jar && \
    curl -L -o /opt/spark/jars/commons-pool2-2.11.1.jar \
    https://repo1.maven.org/maven2/org/apache/commons/commons-pool2/2.11.1/commons-pool2-2.11.1.jar

WORKDIR /app

USER 185
```

### compose.yml에 Spark 서비스 추가

apache/spark 이미지는 `spark-class` 명령어로 Master/Worker를 실행합니다.
`spark-class`는 JVM 설정을 포함해 Spark 클래스를 실행하는 스크립트입니다.

```yaml
# docker/compose.yml에 추가

  # ============================================
  # Phase 4: Spark 분산 처리
  # ============================================
  # spark-class: JVM 설정으로 Spark 클래스를 실행하는 스크립트
  # spark-submit: Spark 애플리케이션을 클러스터에 제출하는 명령어

  spark-master:
    image: apache/spark:3.5.8
    container_name: metro-spark-master
    hostname: spark-master
    command: /opt/spark/bin/spark-class org.apache.spark.deploy.master.Master
    ports:
      - "7077:7077"   # Master 포트 (Worker/Driver 연결용)
      - "4040:4040"   # Spark Application UI (앱 실행 중에만)
      - "8081:8080"   # Master Web UI
    environment:
      SPARK_MODE: master
      SPARK_MASTER_HOST: spark-master
      SPARK_MASTER_PORT: 7077
      SPARK_MASTER_WEBUI_PORT: 8080
    volumes:
      - ../data:/data
      - ../src:/app
    networks:
      - metro-network

  spark-worker:
    image: apache/spark:3.5.8
    container_name: metro-spark-worker
    hostname: spark-worker
    command: /opt/spark/bin/spark-class org.apache.spark.deploy.worker.Worker spark://spark-master:7077
    environment:
      SPARK_MODE: worker
      SPARK_MASTER_URL: spark://spark-master:7077
      SPARK_WORKER_MEMORY: 2g
      SPARK_WORKER_CORES: 2
    depends_on:
      - spark-master
    volumes:
      - ../data:/data
      - ../src:/app
    networks:
      - metro-network

  # Spark 애플리케이션 실행 (profiles 사용)
  # 실행: docker compose --profile spark-job run spark-submit
  spark-submit:
    build:
      context: ..
      dockerfile: docker/Dockerfile.spark
    container_name: metro-spark-submit
    command: >
      /opt/spark/bin/spark-submit
      --master spark://spark-master:7077
      --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0
      src/spark_batch.py
    environment:
      KAFKA_BOOTSTRAP_SERVERS: kafka:9092
      SPARK_MASTER_URL: spark://spark-master:7077
    volumes:
      - ../data:/data
      - ../src:/app
    depends_on:
      - spark-master
      - kafka
    networks:
      - metro-network
    profiles:
      - spark-job
```

---

## Step 2: Spark 배치 처리 실습

먼저 Structured Streaming 전에 **배치 처리**로 Spark DataFrame API에 익숙해집시다.

### 과제 4-2: Spark 배치 처리 코드 작성

`src/spark_batch.py` 파일을 작성하세요.

**요구사항:**
- Phase 1-3에서 수집한 JSON 파일을 읽어서 Spark DataFrame으로 변환
- 역별, 호선별 집계
- 결과를 Parquet 파일로 저장

### 빈칸 채우기

```python
"""
Spark 배치 처리 - 수집된 JSON 파일 분석
"""
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, hour, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


def create_spark_session() -> SparkSession:
    """SparkSession 생성"""
    # TODO: SparkSession 빌더 패턴으로 생성
    spark = SparkSession.builder \
        .appName(________________) \
        .master("local[*]") \
        .getOrCreate()

    # 로그 레벨 설정 (INFO 메시지 줄이기)
    spark.sparkContext.setLogLevel("WARN")

    return spark


def load_json_files(spark: SparkSession, path: str):
    """JSON 파일들을 DataFrame으로 로드"""

    # 스키마 정의 (옵션 - 자동 추론도 가능)
    schema = StructType([
        StructField("station_name", StringType(), True),
        StructField("subway_id", StringType(), True),
        StructField("train_line", StringType(), True),
        StructField("arrival_message", StringType(), True),
        StructField("arrival_time_sec", StringType(), True),
        StructField("received_at", StringType(), True),
        StructField("collected_at", StringType(), True),
    ])

    # TODO: JSON 파일 읽기 (multiLine=True, 배열 형식 JSON 지원)
    # Kafka로 저장한 파일은 messages 배열 안에 데이터가 있음
    raw_df = spark.read \
        .option("multiLine", True) \
        .json(path)

    # messages 배열 펼치기 (explode)
    from pyspark.sql.functions import explode

    # TODO: messages 배열을 행으로 펼치기
    df = raw_df.select(explode(________________).alias("msg"))

    # 중첩 구조 펼치기
    df = df.select(
        col("msg.station_name").alias("station_name"),
        col("msg.subway_id").alias("subway_id"),
        col("msg.train_line").alias("train_line"),
        col("msg.arrival_message").alias("arrival_message"),
        col("msg.arrival_time_sec").cast(IntegerType()).alias("arrival_time_sec"),
        col("msg.received_at").alias("received_at"),
        col("msg.collected_at").alias("collected_at"),
    )

    return df


def analyze_by_station(df):
    """역별 도착 정보 집계"""
    # TODO: station_name으로 그룹화하고 개수 세기
    station_stats = df.groupBy(________________) \
        .agg(
            count("*").alias("arrival_count"),
            avg("arrival_time_sec").alias("avg_wait_sec")
        ) \
        .orderBy(col("arrival_count").desc())

    return station_stats


def analyze_by_line(df):
    """호선별 집계"""
    # 호선 코드 매핑 (UDF 또는 조인)
    line_mapping = {
        "1001": "1호선", "1002": "2호선", "1003": "3호선",
        "1004": "4호선", "1005": "5호선", "1006": "6호선",
        "1007": "7호선", "1008": "8호선", "1009": "9호선",
    }

    from pyspark.sql.functions import udf

    @udf(StringType())
    def get_line_name(subway_id):
        return line_mapping.get(subway_id, "기타")

    df_with_line = df.withColumn("line_name", get_line_name(col("subway_id")))

    # TODO: line_name으로 그룹화하고 집계
    line_stats = df_with_line.groupBy(________________) \
        .agg(
            count("*").alias("total_count"),
            avg("arrival_time_sec").alias("avg_wait_sec")
        ) \
        .orderBy(col("total_count").desc())

    return line_stats


def save_to_parquet(df, path: str):
    """DataFrame을 Parquet 형식으로 저장"""
    # TODO: Parquet 형식으로 저장 (덮어쓰기 모드)
    df.write \
        .mode(________________) \
        .parquet(path)

    print(f"저장 완료: {path}")


if __name__ == "__main__":
    # SparkSession 생성
    spark = create_spark_session()

    print("=" * 60)
    print("Spark 배치 처리 시작")
    print("=" * 60)

    # 데이터 로드
    df = load_json_files(spark, "data/kafka_batch_*.json")

    print(f"\n전체 레코드 수: {df.count()}")
    print("\n데이터 샘플:")
    df.show(5, truncate=False)

    # 역별 분석
    print("\n" + "=" * 60)
    print("역별 도착 정보 집계")
    print("=" * 60)
    station_stats = analyze_by_station(df)
    station_stats.show(10)

    # 호선별 분석
    print("\n" + "=" * 60)
    print("호선별 집계")
    print("=" * 60)
    line_stats = analyze_by_line(df)
    line_stats.show()

    # 결과 저장
    save_to_parquet(station_stats, "output/station_stats")
    save_to_parquet(line_stats, "output/line_stats")

    # SparkSession 종료
    spark.stop()
    print("\nSpark 배치 처리 완료")
```

<details>
<summary>💡 힌트 1: SparkSession 앱 이름</summary>

```python
.appName("MetroBatchAnalysis")
```

</details>

<details>
<summary>💡 힌트 2: explode로 배열 펼치기</summary>

```python
# messages 컬럼이 배열인 경우
df = raw_df.select(explode(col("messages")).alias("msg"))
```

</details>

<details>
<summary>💡 힌트 3: groupBy</summary>

```python
station_stats = df.groupBy("station_name")
line_stats = df_with_line.groupBy("line_name")
```

</details>

<details>
<summary>💡 힌트 4: 저장 모드</summary>

```python
.mode("overwrite")  # 기존 파일 덮어쓰기
```

</details>

<details>
<summary>✅ 모범 답안</summary>

```python
"""
Spark 배치 처리 - 수집된 JSON 파일 분석
"""
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, explode, udf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


def create_spark_session() -> SparkSession:
    spark = SparkSession.builder \
        .appName("MetroBatchAnalysis") \
        .master("local[*]") \
        .getOrCreate()

    spark.sparkContext.setLogLevel("WARN")
    return spark


def load_json_files(spark: SparkSession, path: str):
    raw_df = spark.read \
        .option("multiLine", True) \
        .json(path)

    df = raw_df.select(explode(col("messages")).alias("msg"))

    df = df.select(
        col("msg.station_name").alias("station_name"),
        col("msg.subway_id").alias("subway_id"),
        col("msg.train_line").alias("train_line"),
        col("msg.arrival_message").alias("arrival_message"),
        col("msg.arrival_time_sec").cast(IntegerType()).alias("arrival_time_sec"),
        col("msg.received_at").alias("received_at"),
        col("msg.collected_at").alias("collected_at"),
    )

    return df


def analyze_by_station(df):
    station_stats = df.groupBy("station_name") \
        .agg(
            count("*").alias("arrival_count"),
            avg("arrival_time_sec").alias("avg_wait_sec")
        ) \
        .orderBy(col("arrival_count").desc())

    return station_stats


def analyze_by_line(df):
    line_mapping = {
        "1001": "1호선", "1002": "2호선", "1003": "3호선",
        "1004": "4호선", "1005": "5호선", "1006": "6호선",
        "1007": "7호선", "1008": "8호선", "1009": "9호선",
    }

    @udf(StringType())
    def get_line_name(subway_id):
        return line_mapping.get(subway_id, "기타")

    df_with_line = df.withColumn("line_name", get_line_name(col("subway_id")))

    line_stats = df_with_line.groupBy("line_name") \
        .agg(
            count("*").alias("total_count"),
            avg("arrival_time_sec").alias("avg_wait_sec")
        ) \
        .orderBy(col("total_count").desc())

    return line_stats


def save_to_parquet(df, path: str):
    df.write \
        .mode("overwrite") \
        .parquet(path)
    print(f"저장 완료: {path}")


if __name__ == "__main__":
    spark = create_spark_session()

    print("=" * 60)
    print("Spark 배치 처리 시작")
    print("=" * 60)

    df = load_json_files(spark, "data/kafka_batch_*.json")

    print(f"\n전체 레코드 수: {df.count()}")
    print("\n데이터 샘플:")
    df.show(5, truncate=False)

    print("\n" + "=" * 60)
    print("역별 도착 정보 집계")
    print("=" * 60)
    station_stats = analyze_by_station(df)
    station_stats.show(10)

    print("\n" + "=" * 60)
    print("호선별 집계")
    print("=" * 60)
    line_stats = analyze_by_line(df)
    line_stats.show()

    save_to_parquet(station_stats, "output/station_stats")
    save_to_parquet(line_stats, "output/line_stats")

    spark.stop()
    print("\nSpark 배치 처리 완료")
```

</details>

---

## Step 3: Spark Structured Streaming

### 과제 4-3: Kafka-Spark Streaming 연동

`src/spark_streaming.py` 파일을 작성하세요.

**요구사항:**
- Kafka `metro-arrivals` 토픽에서 실시간으로 데이터 읽기
- JSON 파싱 및 스키마 적용
- 1분 윈도우로 역별 집계
- 콘솔 및 파일로 출력

### 빈칸 채우기

```python
"""
Spark Structured Streaming - Kafka에서 실시간 데이터 처리
"""
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, window, count, avg,
    current_timestamp, to_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)


# Kafka 설정
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
KAFKA_TOPIC = "metro-arrivals"


def create_spark_session() -> SparkSession:
    """Streaming용 SparkSession 생성"""
    spark = SparkSession.builder \
        .appName("MetroStreaming") \
        .master("local[*]") \
        .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoint") \
        .getOrCreate()

    spark.sparkContext.setLogLevel("WARN")
    return spark


def get_message_schema():
    """Kafka 메시지의 JSON 스키마"""
    return StructType([
        StructField("station_name", StringType(), True),
        StructField("subway_id", StringType(), True),
        StructField("train_line", StringType(), True),
        StructField("arrival_message", StringType(), True),
        StructField("arrival_time_sec", StringType(), True),
        StructField("received_at", StringType(), True),
        StructField("collected_at", StringType(), True),
    ])


def read_from_kafka(spark: SparkSession):
    """Kafka에서 스트림 읽기"""

    # TODO: Kafka 소스에서 스트림 읽기
    # format: "kafka"
    # option: kafka.bootstrap.servers, subscribe
    kafka_df = spark.readStream \
        .format(________________) \
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
        .option(________________, KAFKA_TOPIC) \
        .option("startingOffsets", "latest") \
        .load()

    return kafka_df


def parse_messages(kafka_df, schema):
    """Kafka 메시지 파싱"""

    # Kafka 메시지의 value는 바이트 배열이므로 문자열로 변환
    # TODO: value 컬럼을 문자열로 캐스팅
    string_df = kafka_df.selectExpr(________________)

    # TODO: JSON 파싱 (from_json 함수 사용)
    parsed_df = string_df.select(
        from_json(col("value"), schema).alias("data")
    )

    # 중첩 구조 펼치기
    flat_df = parsed_df.select(
        col("data.station_name").alias("station_name"),
        col("data.subway_id").alias("subway_id"),
        col("data.train_line").alias("train_line"),
        col("data.arrival_message").alias("arrival_message"),
        col("data.arrival_time_sec").cast(IntegerType()).alias("arrival_time_sec"),
        to_timestamp(col("data.collected_at")).alias("event_time"),
    ).withColumn("processing_time", current_timestamp())

    return flat_df


def aggregate_by_station(df):
    """역별 실시간 집계 (1분 윈도우)"""

    # TODO: 1분 윈도우로 역별 집계
    # window(이벤트시간컬럼, 윈도우크기)
    aggregated = df \
        .withWatermark("event_time", "1 minute") \
        .groupBy(
            window(________________, "1 minute"),
            "station_name"
        ) \
        .agg(
            count("*").alias("arrival_count"),
            avg("arrival_time_sec").alias("avg_wait_sec")
        )

    return aggregated


def write_to_console(df, output_mode="complete"):
    """콘솔로 출력 (디버깅용)"""

    # TODO: 콘솔 싱크로 스트림 쓰기
    query = df.writeStream \
        .format("console") \
        .outputMode(output_mode) \
        .option("truncate", False) \
        .start()

    return query


def write_to_parquet(df, path: str):
    """Parquet 파일로 저장 (append 모드)"""

    query = df.writeStream \
        .format("parquet") \
        .outputMode("append") \
        .option("path", path) \
        .option("checkpointLocation", f"{path}/_checkpoint") \
        .start()

    return query


if __name__ == "__main__":
    print("=" * 60)
    print("Spark Structured Streaming 시작")
    print(f"Kafka: {KAFKA_BOOTSTRAP_SERVERS}")
    print(f"Topic: {KAFKA_TOPIC}")
    print("=" * 60)

    # SparkSession 생성
    spark = create_spark_session()

    # 스키마 정의
    schema = get_message_schema()

    # Kafka에서 읽기
    kafka_df = read_from_kafka(spark)

    # 메시지 파싱
    parsed_df = parse_messages(kafka_df, schema)

    # 역별 집계
    station_agg = aggregate_by_station(parsed_df)

    # 콘솔 출력 (디버깅)
    console_query = write_to_console(station_agg, "complete")

    # Parquet 저장 (원본 데이터)
    # parquet_query = write_to_parquet(parsed_df, "output/streaming_data")

    print("\nStreaming 시작... (Ctrl+C로 종료)")

    # 스트림 대기
    console_query.awaitTermination()
```

<details>
<summary>💡 힌트 1: Kafka format</summary>

```python
.format("kafka")
```

</details>

<details>
<summary>💡 힌트 2: subscribe 옵션</summary>

```python
.option("subscribe", KAFKA_TOPIC)
```

</details>

<details>
<summary>💡 힌트 3: value를 문자열로 캐스팅</summary>

```python
string_df = kafka_df.selectExpr("CAST(value AS STRING) as value")
```

</details>

<details>
<summary>💡 힌트 4: window 함수</summary>

```python
window(col("event_time"), "1 minute")
```

</details>

<details>
<summary>✅ 모범 답안</summary>

```python
"""
Spark Structured Streaming - Kafka에서 실시간 데이터 처리
"""
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, window, count, avg,
    current_timestamp, to_timestamp
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
KAFKA_TOPIC = "metro-arrivals"


def create_spark_session() -> SparkSession:
    spark = SparkSession.builder \
        .appName("MetroStreaming") \
        .master("local[*]") \
        .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoint") \
        .getOrCreate()

    spark.sparkContext.setLogLevel("WARN")
    return spark


def get_message_schema():
    return StructType([
        StructField("station_name", StringType(), True),
        StructField("subway_id", StringType(), True),
        StructField("train_line", StringType(), True),
        StructField("arrival_message", StringType(), True),
        StructField("arrival_time_sec", StringType(), True),
        StructField("received_at", StringType(), True),
        StructField("collected_at", StringType(), True),
    ])


def read_from_kafka(spark: SparkSession):
    kafka_df = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
        .option("subscribe", KAFKA_TOPIC) \
        .option("startingOffsets", "latest") \
        .load()

    return kafka_df


def parse_messages(kafka_df, schema):
    string_df = kafka_df.selectExpr("CAST(value AS STRING) as value")

    parsed_df = string_df.select(
        from_json(col("value"), schema).alias("data")
    )

    flat_df = parsed_df.select(
        col("data.station_name").alias("station_name"),
        col("data.subway_id").alias("subway_id"),
        col("data.train_line").alias("train_line"),
        col("data.arrival_message").alias("arrival_message"),
        col("data.arrival_time_sec").cast(IntegerType()).alias("arrival_time_sec"),
        to_timestamp(col("data.collected_at")).alias("event_time"),
    ).withColumn("processing_time", current_timestamp())

    return flat_df


def aggregate_by_station(df):
    aggregated = df \
        .withWatermark("event_time", "1 minute") \
        .groupBy(
            window(col("event_time"), "1 minute"),
            "station_name"
        ) \
        .agg(
            count("*").alias("arrival_count"),
            avg("arrival_time_sec").alias("avg_wait_sec")
        )

    return aggregated


def write_to_console(df, output_mode="complete"):
    query = df.writeStream \
        .format("console") \
        .outputMode(output_mode) \
        .option("truncate", False) \
        .start()

    return query


if __name__ == "__main__":
    print("=" * 60)
    print("Spark Structured Streaming 시작")
    print("=" * 60)

    spark = create_spark_session()
    schema = get_message_schema()

    kafka_df = read_from_kafka(spark)
    parsed_df = parse_messages(kafka_df, schema)
    station_agg = aggregate_by_station(parsed_df)

    console_query = write_to_console(station_agg, "complete")

    print("\nStreaming 시작... (Ctrl+C로 종료)")
    console_query.awaitTermination()
```

</details>

---

## Step 4: 실행 및 테스트

### 과제 4-4: Spark Streaming 실행

#### 1. Spark 클러스터 시작

```bash
cd docker

# 전체 인프라 시작
docker compose up -d zookeeper kafka kafka-ui spark-master spark-worker

# 상태 확인
docker compose ps
```

#### 2. Spark Web UI 확인

브라우저에서 http://localhost:8081 접속 (Spark Master UI)

#### 3. 터미널 2개에서 실행

**터미널 1: Spark Streaming (Consumer)**
```bash
docker compose run --rm spark-submit
```

**터미널 2: Kafka Producer**
```bash
docker compose run --rm producer
```

### 예상 출력

<details>
<summary>📋 Spark Streaming 출력 (클릭해서 펼치기)</summary>

```
============================================================
Spark Structured Streaming 시작
Kafka: kafka:29092
Topic: metro-arrivals
============================================================

Streaming 시작... (Ctrl+C로 종료)

-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+------------+-------------+------------+
|window                                    |station_name|arrival_count|avg_wait_sec|
+------------------------------------------+------------+-------------+------------+
|{2024-01-15 16:30:00, 2024-01-15 16:31:00}|강남         |12           |125.5       |
|{2024-01-15 16:30:00, 2024-01-15 16:31:00}|홍대입구     |10           |98.2        |
|{2024-01-15 16:30:00, 2024-01-15 16:31:00}|신도림       |11           |145.8       |
+------------------------------------------+------------+-------------+------------+

-------------------------------------------
Batch: 1
-------------------------------------------
...
```

</details>

---

## Step 5: Git 커밋

### 과제 4-5: Phase 4 완료 커밋

```bash
cd ..

git add .
git commit -m "[Phase 4] Spark 분산 처리 - Driver: 김철수

- Spark Docker 환경 구성 (Master, Worker)
- Spark 배치 처리 구현 (spark_batch.py)
- Spark Structured Streaming 구현 (spark_streaming.py)
- Kafka-Spark 연동 완료
- 역별/호선별 실시간 집계"

git push origin main
```

---

## 체크포인트

Phase 4를 완료하기 전에 다음을 확인하세요:

- [ ] Spark Master/Worker 컨테이너 실행 중
- [ ] http://localhost:8081 에서 Spark UI 접속 가능
- [ ] Spark 배치 처리 (`spark_batch.py`) 실행 성공
- [ ] Spark Streaming + Kafka Producer 연동 성공
- [ ] 콘솔에 역별 집계 결과 출력
- [ ] Git 커밋 완료

---

## Phase 4의 성과와 남은 한계

### 해결된 문제

**대용량 데이터 처리 가능!**

| 데이터 양 | Python Consumer | Spark |
|-----------|-----------------|-------|
| 1,000,000건 | 메모리 부족 | 분산 처리로 OK |

**복잡한 집계 연산 가능:**
- 윈도우 집계 (시간대별)
- 다중 조인
- 복잡한 변환

**실시간 + 배치 모두 지원:**
- Structured Streaming: 실시간
- Batch DataFrame: 배치

### 남은 한계

**아직 수동 실행...**

`$ docker compose run --rm spark-submit` ← 매번 직접 실행해야 함

→ **Phase 5에서 GitHub Actions로 자동화!**

---

## 다음 단계

**Phase 5: GitHub Actions 스케줄링**으로 이동하세요.

> 파일: `05_Phase5_GitHub_Actions.py`

Phase 5에서는 수동 실행의 피로를 GitHub Actions로 해결합니다.
매시간 자동으로 데이터를 수집하는 스케줄링을 구현해봅시다.

---

## FAQ

<details>
<summary><b>Q1. Spark가 Kafka에 연결되지 않아요</b></summary>

**체크리스트:**
1. Kafka가 실행 중인지 확인
2. 네트워크 설정 확인 (같은 Docker 네트워크)
3. `kafka:29092` (컨테이너 내부 주소) 사용 확인

```bash
# Kafka 상태 확인
docker compose logs kafka | tail -20
```

</details>

<details>
<summary><b>Q2. "No module named pyspark" 에러</b></summary>

Dockerfile.spark를 사용하고 있는지 확인하세요.
일반 Python 이미지에는 PySpark가 설치되어 있지 않습니다.

```yaml
spark-submit:
  build:
    dockerfile: docker/Dockerfile.spark  # 확인!
```

</details>

<details>
<summary><b>Q3. Streaming이 아무것도 출력하지 않아요</b></summary>

**원인**: Kafka에 새 메시지가 없음

**해결**:
1. Producer를 실행하여 새 메시지 전송
2. `startingOffsets`를 `earliest`로 변경 (처음부터 읽기)

```python
.option("startingOffsets", "earliest")
```

</details>

<details>
<summary><b>Q4. Parquet 파일이 생성되지 않아요</b></summary>

**체크리스트:**
1. 출력 경로의 쓰기 권한 확인
2. 볼륨 마운트 설정 확인
3. checkpoint 디렉토리 권한 확인

```bash
# 권한 설정
chmod 777 output/
```

</details>

---


# Phase 5: GitHub Actions - 자동화 스케줄링

## 학습 목표
- GitHub Actions를 이용한 워크플로우 자동화 이해
- Cron 표현식을 사용한 스케줄링 설정
- CI/CD 파이프라인의 기초 개념 학습
- 데이터 수집 자동화 구현

## 왜 GitHub Actions인가?

### 현재까지의 한계

**Phase 4까지 구현한 파이프라인:**

**수동 실행** → **Docker Compose Up** → **데이터 수집** → **Kafka** → **Spark**

문제: 매번 사람이 실행해야 함!

### 스케줄러의 필요성

**실제 데이터 파이프라인:**

**스케줄러** → **자동 실행** → **데이터 수집** → **처리** → **저장**

정해진 시간에 자동으로 실행:
- 매시간 데이터 수집
- 매일 새벽 배치 처리
- 매주 리포트 생성

## 스케줄링 도구 비교

| 도구 | 복잡도 | 사용 사례 | 특징 |
|------|--------|-----------|------|
| **Cron (Linux)** | 낮음 | 단순 반복 작업 | OS 내장, 단일 서버 |
| **GitHub Actions** | 낮음 | CI/CD, 간단한 스케줄링 | 무료, 클라우드 기반 |
| **Airflow** | 높음 | 복잡한 DAG 워크플로우 | 의존성 관리, 모니터링 |
| **AWS Step Functions** | 중간 | 서버리스 워크플로우 | AWS 통합, 이벤트 기반 |

> **이번 과제에서는 GitHub Actions를 사용합니다.**
> - 무료로 사용 가능
> - 별도 서버 불필요
> - Git과 자연스럽게 통합
> - Airflow는 이후 수업에서 다룰 예정

## GitHub Actions 기본 개념

### 핵심 용어

```yaml
Workflow (워크플로우)
├── Event (이벤트): 워크플로우를 트리거하는 조건
│   ├── push: 코드 푸시 시
│   ├── pull_request: PR 생성/업데이트 시
│   ├── schedule: 정해진 시간에 (cron)
│   └── workflow_dispatch: 수동 실행
│
├── Job (작업): 병렬 또는 순차 실행되는 단위
│   ├── runs-on: 실행 환경 (ubuntu-latest, etc.)
│   └── steps: 작업 내 단계들
│
└── Step (단계): 실제 명령어 실행
    ├── uses: 액션 사용 (actions/checkout@v4)
    └── run: 쉘 명령어 실행
```

## Cron 표현식 이해하기

### Cron 표현식 형식

| 위치 | 의미 | 범위 |
|------|------|------|
| 1번째 | 분 | 0-59 |
| 2번째 | 시 | 0-23 |
| 3번째 | 일 | 1-31 |
| 4번째 | 월 | 1-12 |
| 5번째 | 요일 | 0-6 (일요일=0) |

예: `* * * * *` (5개 필드)

### 자주 사용하는 예시
```yaml
# 매시간 정각
- cron: '0 * * * *'

# 매일 자정 (UTC)
- cron: '0 0 * * *'

# 매일 오전 9시 (KST = UTC+9, 즉 UTC 0시)
- cron: '0 0 * * *'

# 매주 월요일 오전 9시 (KST)
- cron: '0 0 * * 1'

# 5분마다
- cron: '*/5 * * * *'

# 평일 매시간
- cron: '0 * * * 1-5'
```

> **주의**: GitHub Actions의 cron은 **UTC 기준**입니다!
> - 한국 시간(KST) = UTC + 9시간
> - KST 오전 9시 = UTC 0시

## 실습 5-1: 기본 워크플로우 작성

### 파일 구조
```
metro-pipeline/
├── .github/
│   └── workflows/
│       └── collect-data.yml    # 워크플로우 정의
├── src/
│   └── collector.py            # 데이터 수집 스크립트
└── requirements.txt
```

### Step 1: 간단한 데이터 수집 스크립트

`src/simple_collector.py`:

```python
"""
GitHub Actions에서 실행할 간단한 데이터 수집기
- API 호출 → JSON 저장
- 외부 의존성 최소화
"""
import os
import json
import urllib.request
from datetime import datetime

def collect_metro_data():
    """서울 지하철 실시간 데이터 수집"""

    api_key = os.environ.get("SEOUL_API_KEY", "sample")
    station = "서울역"

    # API URL 구성
    url = f"http://swopenapi.seoul.go.kr/api/subway/{api_key}/json/realtimeStationArrival/0/10/{station}"

    try:
        # API 호출
        with urllib.request.urlopen(url, timeout=30) as response:
            data = json.loads(response.read().decode())

        # 타임스탬프 추가
        result = {
            "collected_at": datetime.now().isoformat(),
            "station": station,
            "data": data
        }

        # 결과 저장
        filename = f"data/metro_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        os.makedirs("data", exist_ok=True)

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"데이터 저장 완료: {filename}")
        print(f"수집된 레코드: {len(data.get('realtimeArrivalList', []))}건")

        return True

    except Exception as e:
        print(f"수집 실패: {e}")
        return False

if __name__ == "__main__":
    success = collect_metro_data()
    exit(0 if success else 1)
```

### Step 2: GitHub Actions 워크플로우 작성

`.github/workflows/collect-data.yml`:

```yaml
# 워크플로우 이름
name: Collect Metro Data

# 트리거 조건
on:
  # 스케줄 실행 (매시간)
  schedule:
    - cron: '0 * * * *'  # 매시간 정각 (UTC)

  # 수동 실행 허용
  workflow_dispatch:
    inputs:
      station:
        description: '수집할 역 이름'
        required: false
        default: '서울역'

# 작업 정의
jobs:
  collect:
    runs-on: ubuntu-latest

    steps:
      # 1. 코드 체크아웃
      - name: Checkout repository
        uses: actions/checkout@v4

      # 2. Python 설정
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      # 3. 의존성 설치
      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install requests

      # 4. 데이터 수집 실행
      - name: Collect data
        env:
          SEOUL_API_KEY: ${{ secrets.SEOUL_API_KEY }}
        run: python src/simple_collector.py

      # 5. 결과 업로드 (Artifact)
      - name: Upload collected data
        uses: actions/upload-artifact@v4
        with:
          name: metro-data-${{ github.run_number }}
          path: data/
          retention-days: 7
```

## 실습 5-2: 빈칸 채우기 - 워크플로우 완성

### 문제: 다음 워크플로우의 빈칸을 채우세요

```yaml
name: Metro Data Pipeline

on:
  schedule:
    # 문제 1: 매일 한국시간 오전 6시에 실행되도록 cron 작성
    # 힌트: KST 06:00 = UTC ??:00
    - cron: '____________'

  workflow_dispatch:

jobs:
  collect-and-process:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Set up Python
        uses: ____________  # 문제 2: Python 설정 액션
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: |
          pip install requests pandas

      - name: Collect data
        env:
          # 문제 3: GitHub Secrets에서 API 키 가져오기
          API_KEY: ${{ ____________ }}
        run: python src/collector.py

      - name: Process data
        run: python src/processor.py

      - name: Upload results
        uses: actions/upload-artifact@v4
        with:
          name: processed-data
          path: output/
          # 문제 4: 아티팩트 보관 기간 (일 단위)
          ____________: 30
```

<details>
<summary><b>힌트 보기</b></summary>

1. **Cron 표현식**: 한국시간(KST)은 UTC+9입니다. KST 06:00 = UTC 21:00 (전날)
2. **Python 액션**: `actions/setup-python@v5`가 최신 버전입니다
3. **Secrets 접근**: `secrets.시크릿이름` 형식으로 접근합니다
4. **보관 기간**: `retention-days` 키를 사용합니다

</details>

<details>
<summary><b>모범 답안</b></summary>

```yaml
on:
  schedule:
    # 문제 1: KST 06:00 = UTC 21:00 (전날)
    - cron: '0 21 * * *'

steps:
  - name: Set up Python
    # 문제 2: Python 설정 액션
    uses: actions/setup-python@v5

  - name: Collect data
    env:
      # 문제 3: Secrets 접근
      API_KEY: ${{ secrets.SEOUL_API_KEY }}

  - name: Upload results
    uses: actions/upload-artifact@v4
    with:
      name: processed-data
      path: output/
      # 문제 4: 보관 기간
      retention-days: 30
```

</details>

## 실습 5-3: Secrets 설정하기

### GitHub Secrets란?
- API 키, 비밀번호 등 민감한 정보를 안전하게 저장
- 워크플로우에서 환경변수로 사용 가능
- 로그에 자동으로 마스킹 처리

### Secrets 설정 방법
```
1. GitHub 저장소 → Settings
2. Secrets and variables → Actions
3. New repository secret
4. Name: SEOUL_API_KEY
5. Secret: [실제 API 키 입력]
6. Add secret
```

### 워크플로우에서 사용
```yaml
env:
  # Secrets 접근
  API_KEY: ${{ secrets.SEOUL_API_KEY }}

  # 일반 변수
  STATION: "서울역"
```

## 실습 5-4: 다중 작업 워크플로우

### 작업 간 의존성 설정

```yaml
name: Data Pipeline

on:
  schedule:
    - cron: '0 */6 * * *'  # 6시간마다
  workflow_dispatch:

jobs:
  # 작업 1: 데이터 수집
  collect:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Collect data
        run: python src/collector.py
      - uses: actions/upload-artifact@v4
        with:
          name: raw-data
          path: data/raw/

  # 작업 2: 데이터 처리 (collect 완료 후 실행)
  process:
    runs-on: ubuntu-latest
    needs: collect  # collect 작업에 의존
    steps:
      - uses: actions/checkout@v4
      - uses: actions/download-artifact@v4
        with:
          name: raw-data
          path: data/raw/
      - name: Process data
        run: python src/processor.py
      - uses: actions/upload-artifact@v4
        with:
          name: processed-data
          path: data/processed/

  # 작업 3: 리포트 생성 (process 완료 후 실행)
  report:
    runs-on: ubuntu-latest
    needs: process
    steps:
      - uses: actions/checkout@v4
      - uses: actions/download-artifact@v4
        with:
          name: processed-data
          path: data/processed/
      - name: Generate report
        run: python src/reporter.py
```

### 의존성 그래프

**collect** → **process** → **report** (순차 실행)

## 실습 5-5: 빈칸 채우기 - 의존성 설정

### 문제: 다음 워크플로우의 의존성을 올바르게 설정하세요

```yaml
name: ETL Pipeline

on:
  workflow_dispatch:

jobs:
  # 1. 추출 (Extract)
  extract:
    runs-on: ubuntu-latest
    steps:
      - run: echo "Extracting data..."

  # 2. 변환 (Transform) - extract 후에 실행
  transform:
    runs-on: ubuntu-latest
    ____________  # 문제 1: extract 작업에 의존하도록 설정
    steps:
      - run: echo "Transforming data..."

  # 3. 적재 (Load) - transform 후에 실행
  load:
    runs-on: ubuntu-latest
    ____________  # 문제 2: transform 작업에 의존하도록 설정
    steps:
      - run: echo "Loading data..."

  # 4. 검증 (Validate) - extract, transform 모두 완료 후 실행
  validate:
    runs-on: ubuntu-latest
    ____________  # 문제 3: 여러 작업에 의존하도록 설정
    steps:
      - run: echo "Validating data..."
```

<details>
<summary><b>모범 답안</b></summary>

```yaml
# 2. 변환 - extract 후에 실행
transform:
  runs-on: ubuntu-latest
  needs: extract  # 문제 1

# 3. 적재 - transform 후에 실행
load:
  runs-on: ubuntu-latest
  needs: transform  # 문제 2

# 4. 검증 - 여러 작업 완료 후 실행
validate:
  runs-on: ubuntu-latest
  needs: [extract, transform]  # 문제 3: 배열로 여러 작업 지정
```

**의존성 그래프:**

- **extract** → **transform** → **load** (순차)
- **validate**: extract, transform 모두 완료 후 실행

</details>

## 실습 5-6: S3 업로드 워크플로우

### AWS와 연동하는 워크플로우

```yaml
name: Collect and Upload to S3

on:
  schedule:
    - cron: '0 */3 * * *'  # 3시간마다

jobs:
  collect-and-upload:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install requests boto3

      - name: Collect data
        env:
          SEOUL_API_KEY: ${{ secrets.SEOUL_API_KEY }}
        run: python src/collector.py

      # AWS 자격 증명 설정
      - name: Configure AWS credentials
        uses: aws-actions/configure-aws-credentials@v4
        with:
          aws-access-key-id: ${{ secrets.AWS_ACCESS_KEY_ID }}
          aws-secret-access-key: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          aws-region: ap-northeast-2

      # S3에 업로드
      - name: Upload to S3
        run: |
          DATE=$(date +%Y/%m/%d)
          aws s3 cp data/ s3://my-data-lake/metro/raw/$DATE/ --recursive
```

> **필요한 Secrets:**
> - `SEOUL_API_KEY`: 서울 공공데이터 API 키
> - `AWS_ACCESS_KEY_ID`: AWS 액세스 키
> - `AWS_SECRET_ACCESS_KEY`: AWS 비밀 키

## 실습 5-7: 전체 파이프라인 워크플로우

### 완성된 데이터 파이프라인 워크플로우

```yaml
name: Metro Data Pipeline

on:
  # 매시간 실행
  schedule:
    - cron: '0 * * * *'

  # 수동 실행
  workflow_dispatch:
    inputs:
      stations:
        description: '수집할 역 (쉼표 구분)'
        required: false
        default: '서울역,강남,홍대입구'

env:
  PYTHON_VERSION: '3.11'
  DATA_DIR: data

jobs:
  # ===== 데이터 수집 =====
  collect:
    runs-on: ubuntu-latest
    outputs:
      collected_files: ${{ steps.collect.outputs.files }}

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Install dependencies
        run: pip install requests

      - name: Collect metro data
        id: collect
        env:
          SEOUL_API_KEY: ${{ secrets.SEOUL_API_KEY }}
          STATIONS: ${{ github.event.inputs.stations || '서울역,강남,홍대입구' }}
        run: |
          python src/collector.py
          echo "files=$(ls data/raw/*.json | wc -l)" >> $GITHUB_OUTPUT

      - name: Upload raw data
        uses: actions/upload-artifact@v4
        with:
          name: raw-data
          path: ${{ env.DATA_DIR }}/raw/

  # ===== 데이터 처리 =====
  process:
    runs-on: ubuntu-latest
    needs: collect

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Install dependencies
        run: pip install pandas

      - name: Download raw data
        uses: actions/download-artifact@v4
        with:
          name: raw-data
          path: ${{ env.DATA_DIR }}/raw/

      - name: Process data
        run: python src/processor.py

      - name: Upload processed data
        uses: actions/upload-artifact@v4
        with:
          name: processed-data
          path: ${{ env.DATA_DIR }}/processed/

  # ===== S3 업로드 (선택적) =====
  upload:
    runs-on: ubuntu-latest
    needs: process
    if: github.event_name == 'schedule'  # 스케줄 실행시에만

    steps:
      - name: Download processed data
        uses: actions/download-artifact@v4
        with:
          name: processed-data
          path: ${{ env.DATA_DIR }}/processed/

      - name: Configure AWS credentials
        uses: aws-actions/configure-aws-credentials@v4
        with:
          aws-access-key-id: ${{ secrets.AWS_ACCESS_KEY_ID }}
          aws-secret-access-key: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          aws-region: ap-northeast-2

      - name: Upload to S3
        run: |
          DATE=$(date +%Y/%m/%d/%H)
          aws s3 sync ${{ env.DATA_DIR }}/processed/ \
            s3://${{ secrets.S3_BUCKET }}/metro/processed/$DATE/

  # ===== 알림 =====
  notify:
    runs-on: ubuntu-latest
    needs: [collect, process]
    if: always()  # 성공/실패 관계없이 실행

    steps:
      - name: Send notification
        run: |
          if [ "${{ needs.process.result }}" == "success" ]; then
            echo "✅ 파이프라인 성공"
          else
            echo "❌ 파이프라인 실패"
          fi
```

## 워크플로우 실행 및 모니터링

### 실행 방법

1. **자동 실행**: 설정된 cron 스케줄에 따라 자동 실행

2. **수동 실행**:
   - GitHub 저장소 → Actions 탭
   - 워크플로우 선택 → Run workflow
   - 필요시 입력값 지정 후 실행

3. **Push 트리거**:
   ```yaml
   on:
     push:
       branches: [main]
       paths:
         - 'src/**'
         - '.github/workflows/**'
   ```

### 실행 결과 확인

**GitHub Actions 페이지에서 확인 가능:**

| Job | Step | 소요 시간 |
|-----|------|-----------|
| **collect** (23s) | Set up Python | 5s |
| | Install dependencies | 8s |
| | Collect metro data | 7s |
| | Upload raw data | 3s |
| **process** (15s) | Download raw data | 2s |
| | Process data | 10s |
| | Upload processed data | 3s |
| **upload** (8s) | Upload to S3 | 5s |

## 실습 5-8: 워크플로우 작성 과제

### 미션: 완전한 데이터 파이프라인 워크플로우 작성

**요구사항:**
1. 매일 한국시간 오전 9시에 자동 실행
2. 수동 실행도 가능하게 설정
3. 다음 작업들을 순차적으로 실행:
   - `collect`: 데이터 수집
   - `validate`: 데이터 검증
   - `transform`: 데이터 변환
   - `save`: 결과 저장
4. 실패 시 알림 작업 실행

### 과제 템플릿

```yaml
name: Daily Data Pipeline

on:
  # TODO: 스케줄 설정 (매일 KST 09:00)

  # TODO: 수동 실행 설정

jobs:
  collect:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: echo "Collecting data..."
      # TODO: 데이터 수집 로직 추가

  validate:
    runs-on: ubuntu-latest
    # TODO: collect 작업에 의존하도록 설정
    steps:
      - run: echo "Validating data..."

  transform:
    runs-on: ubuntu-latest
    # TODO: validate 작업에 의존하도록 설정
    steps:
      - run: echo "Transforming data..."

  save:
    runs-on: ubuntu-latest
    # TODO: transform 작업에 의존하도록 설정
    steps:
      - run: echo "Saving data..."

  notify-failure:
    runs-on: ubuntu-latest
    # TODO: 다른 작업이 실패했을 때만 실행되도록 설정
    steps:
      - run: echo "Pipeline failed!"
```

<details>
<summary><b>모범 답안</b></summary>

```yaml
name: Daily Data Pipeline

on:
  schedule:
    # KST 09:00 = UTC 00:00
    - cron: '0 0 * * *'

  workflow_dispatch:

jobs:
  collect:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install requests
      - run: python src/collector.py
      - uses: actions/upload-artifact@v4
        with:
          name: raw-data
          path: data/

  validate:
    runs-on: ubuntu-latest
    needs: collect
    steps:
      - uses: actions/checkout@v4
      - uses: actions/download-artifact@v4
        with:
          name: raw-data
          path: data/
      - run: python src/validator.py

  transform:
    runs-on: ubuntu-latest
    needs: validate
    steps:
      - uses: actions/checkout@v4
      - uses: actions/download-artifact@v4
        with:
          name: raw-data
          path: data/
      - run: python src/transformer.py
      - uses: actions/upload-artifact@v4
        with:
          name: processed-data
          path: output/

  save:
    runs-on: ubuntu-latest
    needs: transform
    steps:
      - uses: actions/download-artifact@v4
        with:
          name: processed-data
          path: output/
      - run: echo "Saving to storage..."

  notify-failure:
    runs-on: ubuntu-latest
    needs: [collect, validate, transform, save]
    if: failure()  # 앞선 작업 중 하나라도 실패하면 실행
    steps:
      - run: echo "Pipeline failed! Check the logs."
```

</details>

## Phase 5 핵심 정리

### 배운 내용

| 개념 | 설명 |
|------|------|
| **Workflow** | 자동화된 작업 프로세스 정의 |
| **Event** | 워크플로우 실행 조건 (schedule, push, manual) |
| **Job** | 독립적으로 실행되는 작업 단위 |
| **Step** | Job 내의 개별 명령어 |
| **Cron** | 스케줄 실행을 위한 표현식 |
| **Secrets** | 민감한 정보의 안전한 저장 |
| **Artifacts** | 작업 간 데이터 공유 |
| **needs** | 작업 간 의존성 설정 |

### 스케줄링 도구 발전 경로

| 단계 | 도구 | 특징 |
|------|------|------|
| 현재 학습 | GitHub Actions | 간단한 cron, 클라우드 기반, Git 통합, 무료 |
| 다음 학습 | Airflow | 복잡한 DAG, 의존성 관리, 모니터링/재시도, 엔터프라이즈급 |

## Phase 5 → Phase 6 연결

### 지금까지 구현한 것

**GitHub Actions** → **수집** → **처리** → **Artifact 저장** (로컬 저장만 가능)

### Phase 6에서 확장할 것

**GitHub Actions** → **수집** → **처리** → **S3 Data Lake**

S3 Data Lake에서 분기:

| 경로 | 기능 |
|------|------|
| S3 → Athena 쿼리 | SQL로 데이터 조회 |
| S3 → Glue ETL | 데이터 변환 |
| 최종 | 분석/대시보드 |

> **다음 Phase 6에서는:**
> - AWS S3에 데이터 저장
> - Athena로 SQL 쿼리 실행
> - 서버리스 데이터 레이크 구축

## 드라이버/내비게이터 체크포인트

### Phase 5 완료 체크리스트

- [ ] GitHub Actions 워크플로우 YAML 이해
- [ ] Cron 표현식으로 스케줄 설정 가능
- [ ] Secrets를 안전하게 설정하고 사용
- [ ] 작업 간 의존성(needs) 설정 가능
- [ ] Artifacts로 작업 간 데이터 전달
- [ ] 워크플로우 수동 실행 및 모니터링

### 역할 교대 시점
- Phase 5 완료 후 드라이버/내비게이터 역할 교대
- Phase 6 시작 전 지금까지 구현한 내용 리뷰

---

## 부록: 유용한 GitHub Actions 패턴

### 매트릭스 빌드 (여러 환경에서 테스트)
```yaml
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ['3.9', '3.10', '3.11']
    steps:
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
```

### 캐싱 (의존성 설치 시간 단축)
```yaml
- name: Cache pip packages
  uses: actions/cache@v4
  with:
    path: ~/.cache/pip
    key: ${{ runner.os }}-pip-${{ hashFiles('requirements.txt') }}
```

### 조건부 실행
```yaml
- name: Deploy to production
  if: github.ref == 'refs/heads/main'
  run: ./deploy.sh
```

---


# Phase 6: AWS 클라우드 연계 - 서버리스 데이터 레이크

## 학습 목표
- AWS S3를 데이터 레이크로 활용하는 방법 이해
- Athena를 이용한 서버리스 SQL 쿼리 실습
- 클라우드 기반 데이터 파이프라인 아키텍처 이해
- AWS 데이터 서비스 생태계 개념 학습

## 왜 클라우드인가?

### 온프레미스 vs 클라우드

| 구분 | 온프레미스 (로컬) | 클라우드 (AWS) |
|------|------------------|----------------|
| **인프라** | 직접 구축/관리 | 관리형 서비스 |
| **확장성** | 하드웨어 추가 필요 | 클릭 몇 번으로 확장 |
| **비용** | 초기 투자 큼 | 사용한 만큼 지불 |
| **가용성** | 직접 이중화 구성 | 99.99% SLA 보장 |
| **보안** | 자체 관리 | AWS 보안 인증 |

### 현재까지의 한계

**Phase 5까지:**

**로컬 머신** → **Docker** → **데이터 저장** (로컬 디스크)

문제점:
- 디스크 용량 한계
- 분석 도구 제한
- 공유 어려움

### 클라우드로 확장

**Phase 6:**

**GitHub Actions** → **수집/처리** → **S3 Data Lake** → **Athena 쿼리** / **다른 서비스**

장점:
- 서버리스 SQL
- 무제한 용량
- 강력한 분석
- 협업 가능

## AWS 데이터 서비스 개요

### 핵심 서비스 맵

**AWS 데이터 서비스 흐름:**

| 단계 | 서비스 | 역할 |
|------|--------|------|
| **데이터 수집** | Kinesis (스트리밍) | 실시간 데이터 수집 |
| **데이터 저장** | S3 (Data Lake) | 무제한 객체 스토리지 |
| **데이터 처리** | Glue (ETL) | 서버리스 데이터 변환 |
| **데이터 분석** | Athena (서버리스) | S3 데이터 SQL 쿼리 |
| **데이터 웨어하우스** | Redshift (DW) | 대규모 분석 DB |

**Kinesis** → **S3** → **Glue** → **Athena** / **Redshift**

### 서비스별 역할

| 서비스 | 역할 | 특징 | 비유 |
|--------|------|------|------|
| **S3** | 저장 | 무제한 객체 스토리지 | "클라우드 하드디스크" |
| **Athena** | 쿼리 | 서버리스 SQL | "S3용 SQL 에디터" |
| **Glue** | ETL | 서버리스 데이터 변환 | "자동 데이터 가공기" |
| **Redshift** | DW | 페타바이트급 웨어하우스 | "초대형 분석 DB" |
| **Kinesis** | 수집 | 실시간 스트리밍 | "클라우드 Kafka" |

## 실습 6-0: AWS 환경 설정

### AWS CLI 설치 및 설정

```bash
# 1. AWS CLI 설치 확인
aws --version

# 2. AWS 자격 증명 설정 (AWS Academy 또는 IAM 자격증명)
aws configure
# AWS Access Key ID: [발급받은 키]
# AWS Secret Access Key: [발급받은 시크릿]
# Default region name: ap-northeast-2
# Default output format: json

# 3. 설정 확인
aws sts get-caller-identity
```

### 예상 출력

```json
{
    "UserId": "AIDAXXXXXXXXXXXXXXXX",
    "Account": "123456789012",
    "Arn": "arn:aws:iam::123456789012:user/student"
}
```

### S3 버킷 생성

```bash
# 버킷 이름은 전 세계에서 고유해야 합니다
# 본인의 학번이나 이름을 포함하세요
aws s3 mb s3://metro-data-lake-{학번} --region ap-northeast-2

# 생성 확인
aws s3 ls
```

> **주의**: 버킷 이름 규칙
> - 3-63자 사이
> - 소문자, 숫자, 하이픈만 사용
> - 문자나 숫자로 시작/끝나야 함
> - 전 세계에서 고유해야 함

---

## 실습 6-1: S3 데이터 레이크 구축

### S3 버킷 구조 설계

```
s3://metro-data-lake-{학번}/
│
├── raw/                          # 원본 데이터 (수정 불가 원칙)
│   └── metro/
│       └── year=2024/
│           └── month=01/
│               └── day=15/
│                   ├── hour=09/
│                   │   └── data_20240115_0900.json
│                   └── hour=10/
│                       └── data_20240115_1000.json
│
├── processed/                    # 가공된 데이터
│   └── metro/
│       └── aggregated/
│           └── daily_summary.parquet
│
└── analytics/                    # 분석 결과
    └── reports/
        └── weekly_report.csv
```

> **Hive 파티셔닝**: `year=2024/month=01/day=15` 형식으로 저장하면
> Athena에서 자동으로 파티션으로 인식하여 쿼리 성능이 향상됩니다.

### S3 업로드 코드

`src/s3_uploader.py`:

```python
"""
S3 데이터 업로더
- 수집된 데이터를 S3에 업로드
- Hive 스타일 파티셔닝 적용
"""
import os
import json
import boto3
from datetime import datetime

class S3Uploader:
    """S3에 데이터를 업로드하는 클래스"""

    def __init__(self, bucket_name: str):
        """
        Args:
            bucket_name: S3 버킷 이름
        """
        self.bucket_name = bucket_name
        self.s3_client = boto3.client("s3")

    def _get_partition_path(self, timestamp: datetime) -> str:
        """Hive 스타일 파티션 경로 생성"""
        return (
            f"year={timestamp.year}/"
            f"month={timestamp.month:02d}/"
            f"day={timestamp.day:02d}/"
            f"hour={timestamp.hour:02d}"
        )

    def upload_json(self, data: dict, prefix: str = "raw/metro") -> str:
        """
        JSON 데이터를 S3에 업로드

        Args:
            data: 업로드할 데이터
            prefix: S3 키 접두사

        Returns:
            업로드된 S3 URI
        """
        now = datetime.now()
        partition_path = self._get_partition_path(now)
        filename = f"data_{now.strftime('%Y%m%d_%H%M%S')}.json"

        # S3 키 구성
        s3_key = f"{prefix}/{partition_path}/{filename}"

        # 업로드
        self.s3_client.put_object(
            Bucket=self.bucket_name,
            Key=s3_key,
            Body=json.dumps(data, ensure_ascii=False),
            ContentType="application/json"
        )

        s3_uri = f"s3://{self.bucket_name}/{s3_key}"
        print(f"업로드 완료: {s3_uri}")
        return s3_uri

    def upload_file(self, local_path: str, s3_key: str) -> str:
        """
        로컬 파일을 S3에 업로드

        Args:
            local_path: 로컬 파일 경로
            s3_key: S3 키

        Returns:
            업로드된 S3 URI
        """
        self.s3_client.upload_file(local_path, self.bucket_name, s3_key)
        return f"s3://{self.bucket_name}/{s3_key}"


if __name__ == "__main__":
    import os

    # 환경 변수에서 버킷 이름 가져오기
    bucket = os.environ.get("S3_BUCKET", "metro-data-lake-test")

    # 테스트 데이터 업로드
    uploader = S3Uploader(bucket)

    test_data = {
        "station": "서울역",
        "congestion": 75,
        "timestamp": datetime.now().isoformat()
    }

    uri = uploader.upload_json(test_data)
    print(f"테스트 업로드 완료: {uri}")
```

### 실행 방법

```bash
# 환경 변수 설정 후 실행
export S3_BUCKET=metro-data-lake-{학번}
python src/s3_uploader.py
```

### 예상 출력

<details>
<summary>📋 정상 실행 시 출력 (클릭해서 펼치기)</summary>

```
2026-01-23 11:27:22 - INFO - 버킷 존재: metro-data-lake-{학번}
2026-01-23 11:27:22 - INFO - 업로드 완료: s3://metro-data-lake-{학번}/raw/metro/year=2026/month=01/day=23/hour=11/data_20260123_112722.json
2026-01-23 11:27:22 - INFO - 테스트 완료: s3://metro-data-lake-{학번}/raw/metro/year=2026/month=01/day=23/hour=11/data_20260123_112722.json
2026-01-23 11:27:22 - INFO - 버킷 내 객체 수: 1
2026-01-23 11:27:22 - INFO -   - raw/metro/year=2026/month=01/day=23/hour=11/data_20260123_112722.json (211 bytes)
```

</details>

### 업로드 확인

```bash
# AWS CLI로 업로드된 파일 확인
aws s3 ls s3://metro-data-lake-{학번}/raw/metro/ --recursive

# 파일 내용 확인
aws s3 cp s3://metro-data-lake-{학번}/raw/metro/year=2026/month=01/day=23/hour=11/data_20260123_112722.json -
```

---

## 실습 6-2: 빈칸 채우기 - S3 업로드 구현

### 문제: 다음 코드의 빈칸을 채우세요

```python
import boto3
from datetime import datetime

def upload_to_s3(data: dict, bucket: str) -> str:
    """데이터를 S3에 업로드"""

    # 문제 1: S3 클라이언트 생성
    s3 = boto3.____________("s3")

    # 파티션 경로 생성
    now = datetime.now()
    partition = f"year={now.year}/month={now.month:02d}/day={now.day:02d}"

    # S3 키 구성
    key = f"raw/metro/{partition}/data.json"

    # 문제 2: S3에 객체 업로드
    s3.____________(
        Bucket=bucket,
        Key=key,
        Body=json.dumps(data),
        ContentType="application/json"
    )

    return f"s3://{bucket}/{key}"


def download_from_s3(bucket: str, key: str) -> dict:
    """S3에서 데이터 다운로드"""

    s3 = boto3.client("s3")

    # 문제 3: S3에서 객체 가져오기
    response = s3.____________(Bucket=bucket, Key=key)

    # 문제 4: 응답 본문 읽기
    content = response["____________"].read().decode("utf-8")

    return json.loads(content)
```

<details>
<summary><b>힌트 보기</b></summary>

1. **클라이언트 생성**: `boto3.client()` 또는 `boto3.resource()` 중 하나
2. **객체 업로드**: `put_object` 메서드 사용
3. **객체 다운로드**: `get_object` 메서드 사용
4. **응답 본문**: S3 응답에서 파일 내용이 담긴 키

</details>

<details>
<summary><b>모범 답안</b></summary>

```python
# 문제 1: S3 클라이언트 생성
s3 = boto3.client("s3")

# 문제 2: S3에 객체 업로드
s3.put_object(
    Bucket=bucket,
    Key=key,
    Body=json.dumps(data),
    ContentType="application/json"
)

# 문제 3: S3에서 객체 가져오기
response = s3.get_object(Bucket=bucket, Key=key)

# 문제 4: 응답 본문 읽기
content = response["Body"].read().decode("utf-8")
```

</details>

## 실습 6-3: AWS Athena 쿼리

### Athena란?
- S3에 저장된 데이터를 **SQL로 직접 쿼리**
- 서버 프로비저닝 불필요 (서버리스)
- 스캔한 데이터 양에 따라 과금
- Presto/Trino 기반의 분산 쿼리 엔진

### Athena 테이블 생성

```sql
-- Athena에서 S3 데이터를 테이블로 정의

-- 1. 데이터베이스 생성
CREATE DATABASE IF NOT EXISTS metro_analytics;

-- 2. 외부 테이블 생성 (S3 데이터 참조)
CREATE EXTERNAL TABLE IF NOT EXISTS metro_analytics.raw_arrivals (
    station STRING,
    line STRING,
    direction STRING,
    destination STRING,
    arrival_time STRING,
    congestion INT,
    collected_at TIMESTAMP
)
PARTITIONED BY (
    year INT,
    month INT,
    day INT,
    hour INT
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION 's3://metro-data-lake-{학번}/raw/metro/'
TBLPROPERTIES ('has_encrypted_data'='false');

-- 3. 파티션 자동 감지
MSCK REPAIR TABLE metro_analytics.raw_arrivals;
```

### Athena 쿼리 예시

```sql
-- 1. 역별 평균 혼잡도
SELECT
    station,
    AVG(congestion) as avg_congestion,
    COUNT(*) as record_count
FROM metro_analytics.raw_arrivals
WHERE year = 2024 AND month = 1 AND day = 15
GROUP BY station
ORDER BY avg_congestion DESC;

-- 2. 시간대별 혼잡도 추이
SELECT
    hour,
    AVG(congestion) as avg_congestion
FROM metro_analytics.raw_arrivals
WHERE station = '강남'
GROUP BY hour
ORDER BY hour;

-- 3. 노선별 운행 현황
SELECT
    line,
    COUNT(DISTINCT station) as station_count,
    COUNT(*) as total_arrivals
FROM metro_analytics.raw_arrivals
GROUP BY line;
```

## 실습 6-4: Python에서 Athena 쿼리

`src/athena_querier.py`:

```python
"""
Python에서 Athena 쿼리 실행
"""
import boto3
import time
import pandas as pd

class AthenaQuerier:
    """Athena 쿼리 실행 클래스"""

    def __init__(self, database: str, output_bucket: str):
        """
        Args:
            database: Athena 데이터베이스 이름
            output_bucket: 쿼리 결과 저장 버킷
        """
        self.database = database
        self.output_location = f"s3://{output_bucket}/athena-results/"
        self.client = boto3.client("athena")

    def execute_query(self, query: str) -> str:
        """
        쿼리 실행 시작

        Returns:
            query_execution_id
        """
        response = self.client.start_query_execution(
            QueryString=query,
            QueryExecutionContext={"Database": self.database},
            ResultConfiguration={"OutputLocation": self.output_location}
        )
        return response["QueryExecutionId"]

    def wait_for_completion(self, execution_id: str, max_wait: int = 60) -> str:
        """쿼리 완료 대기"""
        for _ in range(max_wait):
            response = self.client.get_query_execution(
                QueryExecutionId=execution_id
            )
            state = response["QueryExecution"]["Status"]["State"]

            if state == "SUCCEEDED":
                return "SUCCEEDED"
            elif state in ["FAILED", "CANCELLED"]:
                reason = response["QueryExecution"]["Status"].get("StateChangeReason", "Unknown")
                raise Exception(f"Query {state}: {reason}")

            time.sleep(1)

        raise TimeoutError("Query timed out")

    def get_results(self, execution_id: str) -> pd.DataFrame:
        """쿼리 결과를 DataFrame으로 반환"""
        response = self.client.get_query_results(
            QueryExecutionId=execution_id
        )

        # 컬럼 이름 추출
        columns = [
            col["Label"]
            for col in response["ResultSet"]["ResultSetMetadata"]["ColumnInfo"]
        ]

        # 데이터 행 추출 (첫 행은 헤더)
        rows = []
        for row in response["ResultSet"]["Rows"][1:]:
            rows.append([field.get("VarCharValue", "") for field in row["Data"]])

        return pd.DataFrame(rows, columns=columns)

    def query(self, sql: str) -> pd.DataFrame:
        """쿼리 실행하고 결과 반환"""
        execution_id = self.execute_query(sql)
        self.wait_for_completion(execution_id)
        return self.get_results(execution_id)


# 사용 예시
if __name__ == "__main__":
    querier = AthenaQuerier(
        database="metro_analytics",
        output_bucket="metro-data-lake-test"
    )

    # 역별 평균 혼잡도 조회
    df = querier.query("""
        SELECT station, AVG(congestion) as avg_congestion
        FROM raw_arrivals
        WHERE year = 2024 AND month = 1
        GROUP BY station
        ORDER BY avg_congestion DESC
        LIMIT 10
    """)

    print(df)
```

## 실습 6-5: 빈칸 채우기 - Athena 쿼리 실행

### 문제: Athena 쿼리 실행 코드 완성

```python
import boto3

def run_athena_query(query: str, database: str, output_bucket: str):
    """Athena 쿼리 실행"""

    athena = boto3.client("athena")

    # 문제 1: 쿼리 실행 시작
    response = athena.____________(
        QueryString=query,
        QueryExecutionContext={"Database": database},
        ResultConfiguration={
            "OutputLocation": f"s3://{output_bucket}/results/"
        }
    )

    execution_id = response["____________"]  # 문제 2: 실행 ID 추출

    # 완료 대기
    while True:
        # 문제 3: 쿼리 실행 상태 확인
        status = athena.____________(
            QueryExecutionId=execution_id
        )

        state = status["QueryExecution"]["Status"]["State"]

        if state == "SUCCEEDED":
            break
        elif state in ["FAILED", "CANCELLED"]:
            raise Exception(f"Query failed: {state}")

        time.sleep(1)

    # 문제 4: 쿼리 결과 가져오기
    results = athena.____________(QueryExecutionId=execution_id)

    return results
```

<details>
<summary><b>모범 답안</b></summary>

```python
# 문제 1: 쿼리 실행 시작
response = athena.start_query_execution(

# 문제 2: 실행 ID 추출
execution_id = response["QueryExecutionId"]

# 문제 3: 쿼리 실행 상태 확인
status = athena.get_query_execution(

# 문제 4: 쿼리 결과 가져오기
results = athena.get_query_results(QueryExecutionId=execution_id)
```

</details>

## AWS 서비스 심화 개념

### AWS Glue - 서버리스 ETL

**Glue의 역할:**

**S3 원본 데이터** → **Glue Crawler** → **Data Catalog** (스키마 자동 감지)

**Data Catalog** → **Glue ETL Job** → **S3 가공 데이터**

| Glue ETL Job 기능 |
|-------------------|
| 데이터 변환 |
| 포맷 변경 (JSON → Parquet) |
| 데이터 정제 |

### AWS Redshift - 데이터 웨어하우스

**Redshift vs Athena:**

| 항목 | Athena | Redshift |
|------|--------|----------|
| **서버 관리** | 서버리스 | 클러스터 관리 |
| **비용 모델** | 쿼리당 과금 | 시간당 과금 |
| **성능** | 애드혹 쿼리 | 대규모 분석 |
| **데이터 위치** | S3 (외부) | 자체 스토리지 |
| **용도** | 탐색적 분석 | BI, 대시보드 |

### AWS Kinesis - 실시간 스트리밍

**Kinesis vs Kafka:**

| 항목 | Kinesis | Kafka |
|------|---------|-------|
| **관리** | 완전 관리형 | 자체 관리 |
| **확장성** | 샤드 단위 | 파티션 단위 |
| **보존 기간** | 최대 7일 | 무제한 |
| **AWS 통합** | 네이티브 | 커넥터 필요 |
| **비용** | 샤드/시간 | 인프라 비용 |

**선택 기준:**
- AWS 환경 + 관리형 원함 → **Kinesis**
- 멀티클라우드 + 유연성 → **Kafka**

## 실습 6-6: 전체 파이프라인 통합

### 완성된 아키텍처

**서울 지하철 혼잡도 모니터링 파이프라인**

| Phase | 구성 요소 | 역할 |
|-------|-----------|------|
| **Phase 1-2** | Seoul Metro API → Docker Collector | 데이터 수집 |
| **Phase 3-4** | Kafka (버퍼링) → Spark (분산처리) | 메시지 큐 + 처리 |
| **Phase 5-6** | GitHub Actions (스케줄링) → S3 (Data Lake) | 자동화 + 저장 |

**전체 데이터 흐름:**

**Seoul Metro API** → **Docker Collector** → **Kafka** → **Spark** → **S3**

**S3** → **Athena (쿼리)** + **Glue (ETL)** → **Elasticsearch/Kibana** (다음 수업)

### GitHub Actions + S3 연동 워크플로우

`.github/workflows/pipeline.yml`:

```yaml
name: Metro Data Pipeline to S3

on:
  schedule:
    - cron: '0 * * * *'  # 매시간
  workflow_dispatch:

jobs:
  collect:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install requests boto3

      - name: Collect and upload to S3
        env:
          SEOUL_API_KEY: ${{ secrets.SEOUL_API_KEY }}
          AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
          AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          AWS_REGION: ap-northeast-2
          S3_BUCKET: ${{ secrets.S3_BUCKET }}
        run: |
          python src/collector.py
          python src/s3_uploader.py

  process:
    runs-on: ubuntu-latest
    needs: collect
    steps:
      - uses: actions/checkout@v4

      - name: Configure AWS credentials
        uses: aws-actions/configure-aws-credentials@v4
        with:
          aws-access-key-id: ${{ secrets.AWS_ACCESS_KEY_ID }}
          aws-secret-access-key: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          aws-region: ap-northeast-2

      - name: Run Athena query
        run: |
          # Athena 테이블 파티션 갱신
          aws athena start-query-execution \
            --query-string "MSCK REPAIR TABLE metro_analytics.raw_arrivals" \
            --query-execution-context Database=metro_analytics \
            --result-configuration OutputLocation=s3://${{ secrets.S3_BUCKET }}/athena-results/
```

## Phase 6 핵심 정리

### 배운 내용

| 개념 | 설명 |
|------|------|
| **S3** | 무제한 객체 스토리지, 데이터 레이크의 기반 |
| **파티셔닝** | Hive 스타일 파티션으로 쿼리 성능 향상 |
| **Athena** | S3 데이터를 직접 SQL로 쿼리 (서버리스) |
| **Glue** | 서버리스 ETL, 스키마 자동 감지 |
| **Redshift** | 대규모 데이터 웨어하우스 |
| **Kinesis** | AWS 관리형 실시간 스트리밍 |

### 데이터 레이크 vs 데이터 웨어하우스

| 구분 | 데이터 레이크 (S3) | 데이터 웨어하우스 (Redshift) |
|------|-------------------|------------------------------|
| **데이터 형식** | 원본 그대로 (다양) | 정제된 스키마 |
| **사용자** | 데이터 과학자 | 비즈니스 분석가 |
| **목적** | 탐색, ML | BI, 리포팅 |
| **비용** | 저렴 (저장 중심) | 비쌈 (연산 중심) |

## 다음 수업 연계: Elasticsearch + Kibana

### 이번 과제에서 만든 것

**API** → **Kafka** → **Spark** → **S3** → **Athena**

- SQL로 분석 가능
- But, 실시간 대시보드는?

### 다음 수업에서 배울 것

**Spark/Kafka** → **Elasticsearch** (실시간 검색/집계 엔진) → **Kibana** (실시간 대시보드)

| 장점 |
|------|
| 실시간 데이터 시각화 |
| 복잡한 집계 쿼리 |
| 대시보드 공유 |
| 알림 설정 |

> **미리보기**: 다음 수업에서는 이번에 만든 파이프라인의 출력을
> Elasticsearch에 저장하고 Kibana로 실시간 대시보드를 만듭니다.

## 종합 과제: 전체 파이프라인 구현

### 최종 미션

지금까지 배운 모든 Phase를 통합하여 완전한 데이터 파이프라인을 구축하세요.

### 체크리스트

**Phase 1: 기초 파이프라인**
- [ ] 서울 지하철 API 호출 구현
- [ ] 데이터 수집 및 로컬 저장
- [ ] pandas로 기본 분석

**Phase 2: Docker 컨테이너화**
- [ ] Dockerfile 작성
- [ ] docker-compose.yml 작성
- [ ] 볼륨 마운트로 데이터 영속성 확보

**Phase 3: Kafka 메시지 큐**
- [ ] Kafka Producer 구현
- [ ] Kafka Consumer 구현
- [ ] 데이터 버퍼링 및 처리 분리

**Phase 4: Spark 분산처리**
- [ ] Spark 배치 처리 구현
- [ ] Spark Structured Streaming 구현
- [ ] DataFrame API로 데이터 변환

**Phase 5: GitHub Actions**
- [ ] 워크플로우 YAML 작성
- [ ] Cron 스케줄 설정
- [ ] Secrets 설정 (API 키, AWS 자격증명)

**Phase 6: AWS 연계**
- [ ] S3 버킷 생성 및 구조 설계
- [ ] 데이터 업로드 구현
- [ ] Athena 테이블 생성 및 쿼리

## 페어 프로그래밍 최종 점검

### 드라이버/내비게이터 체크포인트

**Phase 1-3 완료 후 역할 교대**
- 첫 번째 드라이버: 기초 파이프라인 ~ Kafka 구현
- 첫 번째 내비게이터: 코드 리뷰, 힌트 제공

**Phase 4-6 완료 후 역할 교대**
- 두 번째 드라이버: Spark ~ AWS 구현
- 두 번째 내비게이터: 아키텍처 검토, 문서화

### 최종 발표 준비

1. **아키텍처 다이어그램** 준비
2. **각 Phase의 핵심 코드** 설명
3. **겪었던 문제와 해결 방법** 공유
4. **배운 점과 개선할 점** 정리

## 과제 제출 가이드

### 프로젝트 구조
```
metro-pipeline/
├── .github/
│   └── workflows/
│       └── pipeline.yml
├── docker/
│   ├── Dockerfile.python
│   ├── Dockerfile.spark
│   └── compose.yml
├── src/
│   ├── collector.py
│   ├── producer.py
│   ├── consumer.py
│   ├── spark_batch.py
│   ├── spark_streaming.py
│   └── s3_uploader.py
├── sql/
│   └── athena_ddl.sql
├── data/                    # .gitignore
├── requirements.txt
└── README.md
```

### README.md 포함 내용
1. 프로젝트 개요
2. 아키텍처 다이어그램
3. 실행 방법 (로컬, Docker, AWS)
4. 환경 변수 설정
5. 팀원 역할 및 기여

---

## 축하합니다!

이 과제를 완료하면 다음을 달성한 것입니다:

- 실제 API에서 데이터를 수집하는 **데이터 엔지니어링** 경험
- Docker를 활용한 **컨테이너화** 역량
- Kafka를 이용한 **메시지 기반 아키텍처** 이해
- Spark를 활용한 **분산 데이터 처리** 능력
- GitHub Actions를 통한 **자동화** 구현
- AWS 서비스를 활용한 **클라우드 데이터 레이크** 구축

### 이후 학습 경로

| 현재 | 다음 단계 |
|------|-----------|
| GitHub Actions | → Airflow (워크플로우 오케스트레이션) |
| S3 + Athena | → Elasticsearch + Kibana (실시간 시각화) |
| Spark Batch | → Spark ML (머신러닝 파이프라인) |
| 단일 클러스터 | → Kubernetes (컨테이너 오케스트레이션) |

**다음 수업에서 만나요!**